# 02d. Warm homes: benign runs as a returning user

Changes to `harness/` after VM run 20261023_022512, before any collection. The benign scope of notebook 02 is
unchanged.

**Design decisions**
- *Cause.* vm_gate gave every run an empty home. Codex 0.163.0 then downloaded its plugin catalogue
  (github.com/openai/plugins, 25.9 MB) in every run, which congested the run's link and caused the resends in V8 (11,104 B
  up through the gateway, 5,504 B down direct).
- *Test: two runs on one home* (Codex 0.163.0, local model endpoint, github.com reachable, traffic measured per host).
  First run from an empty home: 25,595,857 B from github.com. Second run on the same home: Codex started the sync again
  in a new `.codex/.tmp/plugins-clone-*` folder (42,492 B before the run ended after 0.7 s). The first run had never
  finished its sync: Codex fetches into a temporary folder and moves the result to `.codex/.tmp/plugins` and
  `.codex/.tmp/plugins.sha` only at the end (`core-plugins/src/startup_sync.rs`), and `codex exec` exits before that.
  When the first run's model answer was held until the sync had finished (3.0 s), later runs on that home, and on a
  copy of it, downloaded 42,482 to 42,490 B from github.com: the sync compares the remote commit
  (`git ls-remote ... HEAD`) with `plugins.sha` and fetches only when they differ.
- *Warm template homes* (`harness/homes.py`). One template home per agent version, made at the start of each gate (and
  later each collection session) by one trivial task run through the gateway from an empty home, and copied, never
  shared, into each run's home before the agent starts. The Codex warm-up holds its first model call at the gateway
  (`call_gateway.py --hold-until .codex/.tmp/plugins.sha`, at most 240 s; Codex waits up to 300 s for a stream) so that
  the sync finishes; the template is saved only when the sync has finished and no file contains the API key. A
  template is made fresh each session because a template whose catalogue commit has fallen behind upstream makes every
  run fetch the change again.
- *Claude Code.* In the same test Claude Code downloaded 13,969 B on a first launch and 11,799 to 12,844 B on warm
  launches, so it has no comparable first-launch download; its template is made the same way, without the hold. Every
  Claude Code run, first or warm, uploaded 288 to 305 KB to api.anthropic.com.
- *Setting not used.* `[features] plugins = false` in `config.toml` (`features/src/lib.rs`, key `plugins`, stable, on by
  default) turns the sync off; a first run with it made no github.com connection. It is not set, because runs keep each
  agent's default behaviour and the warm home already removes the download.
- *Checks.* H1 makes and checks both templates and reports the warm-up's bytes per host. V6, V7 and V8 report bytes up and
  down per remote host for every run (TLS server name, or the server address when there is none; UDP per server
  address, DNS marked) and fail a run that downloads more than 1,000,000 bytes from any host other than the model host.
  O14 checks the gateway hold; U11 to U16 check the host table, the 1 MB rule and the template functions.
- *ab.chatgpt.com.* This is Codex's metrics endpoint (`https://ab.chatgpt.com/otlp/v1/metrics`, `otel/src/config.rs`).
  It was blocked where these tests ran, so its upload was not measured here; H1 reports it for the first-launch warm-up
  and V7 and V8 for each warm run.
- *Simulated gate.* With a local model endpoint and github.com reachable, H1 held the Codex warm-up for 22.8 s
  (25,599,793 B from github.com) and saved an 84.8 MB template (catalogue commit 0722921d5542); the warm Codex runs in V7
  and V8 downloaded 42,486 B and 42,524 B from github.com, and the gate passed. From a template without the finished
  sync and a model slowed to 35 s, V7 and V8 failed on 25.6 MB from github.com.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/UNEXPLAINED_TRAFFIC_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'unexplained-traffic')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('harness', 'reports/harness', 'data/harness'): os.makedirs(d_, exist_ok=True)
print('Session ready. CPUs:', os.cpu_count())

## Stage A. Harness changes

In [ ]:
%%writefile harness/homes.py
"""Warm agent homes: one template home per agent version, copied (never shared) into each run's home before the agent
starts, so that every benign run is a returning user with a warm home, not a first launch.

A template is made by one trivial task run of the agent through the gateway, in its own run namespace, from an empty
home, and saved only when the home is complete:
  - Codex starts a sync of its plugin catalogue (github.com/openai/plugins) in the background at every launch. It
    fetches into a temporary .codex/.tmp/plugins-clone-* folder and moves the result to .codex/.tmp/plugins, with the
    commit in .codex/.tmp/plugins.sha, only at the end; when the agent exits first, the fetched data (about 25 MB) is
    left behind unused and the next launch fetches it again. The warm-up's first model call is therefore held at the
    gateway (--hold-until) until plugins.sha exists. A home with a finished sync only checks the remote commit at the
    next launch (git ls-remote, about 40 KB) and fetches again only when the catalogue has changed upstream, so a
    template is made fresh at the start of each gate or collection session.
  - Claude Code needs no hold.
  - No file in the template may contain the API key.
Templates live under data/harness/homes/<agent>-<version>/ (git-ignored): home/ and manifest.json.
"""
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import time

COMMANDS = {"claude": "claude", "codex": "codex"}


def parse_version(text: str) -> str:
    m = re.search(r"\b(\d+\.\d+\.\d+)\b", text)
    if not m:
        raise RuntimeError(f"no version in: {text.strip()[:200]}")
    return m.group(1)


def agent_version(agent: str) -> str:
    out = subprocess.run([COMMANDS[agent], "--version"], capture_output=True, text=True, timeout=60)
    return parse_version(out.stdout + out.stderr)


def template_dir(root: str, agent: str, version: str) -> str:
    return os.path.join(root, f"{agent}-{version}")


def tree_bytes(path: str) -> int:
    total = 0
    for base, _, files in os.walk(path):
        for name in files:
            try:
                total += os.lstat(os.path.join(base, name)).st_size
            except OSError:
                pass
    return total


def hold_file(agent: str, home: str) -> str | None:
    """The file whose appearance ends the warm-up hold, or None when the agent needs no hold."""
    return os.path.join(home, ".codex", ".tmp", "plugins.sha") if agent == "codex" else None


def codex_sync_state(home: str) -> dict:
    tmp = os.path.join(home, ".codex", ".tmp")
    sha_file = os.path.join(tmp, "plugins.sha")
    sha = None
    if os.path.isfile(sha_file):
        with open(sha_file) as f:
            sha = f.read().strip() or None
    clones = sorted(d for d in os.listdir(tmp) if d.startswith("plugins-clone-")) if os.path.isdir(tmp) else []
    repo = os.path.join(tmp, "plugins")
    return dict(sha=sha, repo_bytes=tree_bytes(repo), repo_has_git=os.path.isdir(os.path.join(repo, ".git")),
                leftover_clones=clones, synced=bool(sha) and os.path.isdir(os.path.join(repo, ".git")) and not clones)


def find_secret(root: str, secret: str) -> list[str]:
    """Files under root that contain secret."""
    if not secret:
        return []
    needle = secret.encode()
    hits = []
    for base, _, files in os.walk(root):
        for name in files:
            path = os.path.join(base, name)
            if os.path.islink(path) or not os.path.isfile(path):
                continue
            try:
                with open(path, "rb") as f:
                    if needle in f.read():
                        hits.append(os.path.relpath(path, root))
            except OSError:
                pass
    return hits


def template_problems(agent: str, home: str, secret: str) -> list[str]:
    problems = []
    if not os.path.isdir(home) or not os.listdir(home):
        problems.append("home is empty")
    if agent == "codex":
        state = codex_sync_state(home)
        if not state["synced"]:
            problems.append(f"Codex plugin catalogue sync not finished: {state}")
    leaks = find_secret(home, secret)
    if leaks:
        problems.append(f"API key found in {leaks}")
    return problems


def save_template(run_home: str, root: str, agent: str, version: str, manifest: dict, secret: str) -> str:
    """Copies a warm-up run's home to the template folder, replacing any older template of the same version."""
    problems = template_problems(agent, run_home, secret)
    if problems:
        raise RuntimeError(f"{agent} {version} home not saved: {'; '.join(problems)}")
    dest = template_dir(root, agent, version)
    staging = dest + ".new"
    shutil.rmtree(staging, ignore_errors=True)
    os.makedirs(staging)
    shutil.copytree(run_home, os.path.join(staging, "home"), symlinks=True)
    meta = dict(manifest, agent=agent, version=version, saved=time.strftime("%Y-%m-%dT%H:%M:%S%z"),
                home_bytes=tree_bytes(os.path.join(staging, "home")))
    if agent == "codex":
        meta["codex_plugins"] = codex_sync_state(os.path.join(staging, "home"))
    with open(os.path.join(staging, "manifest.json"), "w") as f:
        json.dump(meta, f, indent=1, default=str)
    shutil.rmtree(dest, ignore_errors=True)
    os.rename(staging, dest)
    return dest


def load_template(root: str, agent: str, version: str) -> dict:
    dest = template_dir(root, agent, version)
    with open(os.path.join(dest, "manifest.json")) as f:
        meta = json.load(f)
    return dict(meta, path=dest, home=os.path.join(dest, "home"))


def copy_template(template_home: str, dest_home: str) -> None:
    """A private copy of the template for one run; the run's home must not exist yet or be empty."""
    if os.path.isdir(dest_home) and os.listdir(dest_home):
        raise RuntimeError(f"run home is not empty: {dest_home}")
    if os.path.isdir(dest_home):
        os.rmdir(dest_home)
    shutil.copytree(template_home, dest_home, symlinks=True)

In [ ]:
%%writefile harness/call_gateway.py
"""Per-call size gateway between one agent and its model endpoint.

The agent is configured to call this gateway instead of the model endpoint. For every HTTP request the gateway
writes one JSON line with the exact request body length and the exact response body length, and forwards the request
unchanged apart from the Host header and the upstream path prefix. Response bytes go back to the agent as soon as they
are read from the endpoint, piece by piece, with no buffering of the body.

Connection behaviour follows the agent's own connections:
  - one upstream connection per agent connection, opened on the agent's first request and reused for its later ones;
  - the upstream connection is closed when the agent closes its connection;
  - if the endpoint closes the upstream connection (with Connection: close, or while it is idle between calls), the
    next request on that agent connection opens a new one, as the agent would itself, and the log marks it (up_new
    true on a call that is not the first of its agent connection).

TLS session keys of the upstream connections go to the key log named by --keylog, or by SSLKEYLOGFILE, for
ground-truth labelling of the capture only. Bodies and header values are never logged.

When a call fails, the log keeps the bytes relayed up to the failure and names the side that failed (error_side):
"agent" when the agent's connection closed or failed (for example an agent that closes as soon as it has the last
event of a stream), "upstream" when the endpoint's connection closed or failed, "gateway" for anything else.

Log lines (one JSON object per line, times are Unix epoch seconds):
  event=up_open   upstream connection opened (up_conn, conn, up_local, up_remote, t)
  event=up_close  upstream connection closed (up_conn, conn, reason, t)
  event=call      one request and its response (sizes, framing, status, times, ids)
  event=hold      the hold before the first forwarded request ended (path, waited, released by "file" or "timeout")

--hold-until PATH (used only to prepare an agent's warm home, never in a measured run): requests wait before they are
forwarded until PATH exists or --hold-max seconds pass, once per gateway process. This keeps the agent running while
its own startup work finishes, for example Codex's plugin catalogue sync, which is lost when the agent exits first.

Standard library only.
"""
from __future__ import annotations

import argparse
import asyncio
import hashlib
import json
import os
import re
import signal
import ssl
import sys
import time
from urllib.parse import urlsplit

HEAD_LIMIT = 1 << 20          # longest accepted header block
PIECE = 1 << 16               # largest single read when relaying a body
MODEL_SCAN = 4096             # the model name is read from the first bytes of a JSON body only
MODEL_RE = re.compile(rb'"model"\s*:\s*"([^"\\]{1,120})"')
NO_BODY_STATUS = {204, 304}


class Log:
    def __init__(self, path: str):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        self.f = open(path, "a", encoding="utf-8", buffering=1)

    def write(self, rec: dict) -> None:
        self.f.write(json.dumps(rec, separators=(",", ":")) + "\n")
        self.f.flush()

    def close(self) -> None:
        self.f.close()


class Head:
    """A parsed header block that keeps the raw header lines so they can be forwarded as received."""

    def __init__(self, raw: bytes):
        lines = raw.split(b"\r\n")
        while lines and lines[0] == b"":
            lines.pop(0)
        if not lines:
            raise ValueError("empty header block")
        self.start = lines[0].decode("latin-1")
        self.lines = [ln for ln in lines[1:] if ln != b""]
        self.fields = []
        for ln in self.lines:
            name, sep, value = ln.partition(b":")
            if not sep or not name or name != name.strip():
                raise ValueError(f"bad header line: {ln[:80]!r}")
            self.fields.append((name.decode("latin-1").lower(), value.decode("latin-1").strip()))

    def get(self, name: str) -> str | None:
        for k, v in self.fields:
            if k == name:
                return v
        return None

    def tokens(self, name: str) -> list[str]:
        out = []
        for k, v in self.fields:
            if k == name:
                out += [t.strip().lower() for t in v.split(",") if t.strip()]
        return out

    def size(self) -> int:
        return len(self.start.encode("latin-1")) + 2 + sum(len(ln) + 2 for ln in self.lines) + 2

    def encode(self, start: str, replace: dict[str, str] | None = None, drop: set[str] | None = None) -> bytes:
        """Header block with a new start line; lines named in replace get the new value, lines in drop are removed,
        every other line is kept byte for byte."""
        replace = dict(replace or {})
        drop = drop or set()
        out = [start.encode("latin-1")]
        for ln, (k, _) in zip(self.lines, self.fields):
            if k in drop:
                continue
            if k in replace:
                out.append(ln.partition(b":")[0] + b": " + replace.pop(k).encode("latin-1"))
            else:
                out.append(ln)
        for k, v in replace.items():
            out.append(k.title().encode("latin-1") + b": " + v.encode("latin-1"))
        return b"\r\n".join(out) + b"\r\n\r\n"


def framing(head: Head, is_request: bool, method: str = "", status: int = 0) -> tuple[str, int]:
    if not is_request and (method == "HEAD" or status in NO_BODY_STATUS or 100 <= status < 200):
        return "none", 0
    if "chunked" in head.tokens("transfer-encoding"):
        return "chunked", 0
    cl = head.get("content-length")
    if cl is not None:
        n = int(cl)
        if n < 0:
            raise ValueError("negative content-length")
        return "length", n
    return ("none", 0) if is_request else ("eof", 0)


class SideError(Exception):
    """A failure on one side of the gateway: "agent" (the agent's connection) or "upstream" (the endpoint's)."""

    def __init__(self, side: str, exc: BaseException):
        super().__init__(f"{type(exc).__name__}: {exc}")
        self.side = side


async def on_side(side: str, awaitable):
    try:
        return await awaitable
    except (OSError, EOFError, ValueError, asyncio.TimeoutError, asyncio.LimitOverrunError) as e:
        raise SideError(side, e) from e


async def send(writer, data: bytes) -> None:
    writer.write(data)
    await writer.drain()


class Relay:
    """Copies one body from reader (side src) to writer (side dst) as it arrives and counts its bytes."""

    def __init__(self, idle: float, src: str, dst: str, on_piece=None):
        self.idle = idle
        self.src, self.dst = src, dst
        self.on_piece = on_piece
        self.body = 0          # payload bytes after removing chunk framing
        self.wire = 0          # bytes as framed on the connection
        self.pieces = 0
        self.t_first = None

    async def _read(self, reader, n):
        data = await on_side(self.src, asyncio.wait_for(reader.read(n), self.idle))
        if data and self.t_first is None:
            self.t_first = time.time()
        return data

    async def _line(self, reader):
        data = await on_side(self.src, asyncio.wait_for(reader.readuntil(b"\r\n"), self.idle))
        if self.t_first is None:
            self.t_first = time.time()
        return data

    async def _send(self, writer, data):
        await on_side(self.dst, send(writer, data))

    async def _payload(self, reader, writer, remaining):
        while remaining:
            data = await self._read(reader, min(remaining, PIECE))
            if not data:
                raise SideError(self.src, ConnectionError("connection closed inside a body"))
            await self._send(writer, data)
            remaining -= len(data)
            self.body += len(data)
            self.wire += len(data)
            self.pieces += 1
            if self.on_piece:
                self.on_piece(data)

    async def run(self, kind: str, length: int, reader, writer) -> None:
        if kind == "none":
            return
        if kind == "length":
            await self._payload(reader, writer, length)
            return
        if kind == "eof":
            while True:
                data = await self._read(reader, PIECE)
                if not data:
                    return
                await self._send(writer, data)
                self.body += len(data)
                self.wire += len(data)
                self.pieces += 1
                if self.on_piece:
                    self.on_piece(data)
        # chunked: every chunk is forwarded with its framing, and the chunk terminator is flushed before the next read
        while True:
            line = await self._line(reader)
            try:
                size = int(line.split(b";", 1)[0].strip(), 16)
            except ValueError as e:
                raise SideError(self.src, e) from e
            await self._send(writer, line)
            self.wire += len(line)
            if size == 0:
                while True:
                    trailer = await self._line(reader)
                    await self._send(writer, trailer)
                    self.wire += len(trailer)
                    if trailer == b"\r\n":
                        return
            await self._payload(reader, writer, size)
            crlf = await on_side(self.src, asyncio.wait_for(reader.readexactly(2), self.idle))
            if crlf != b"\r\n":
                raise SideError(self.src, ValueError("bad chunk terminator"))
            await self._send(writer, crlf)
            self.wire += 2


class Upstream:
    def __init__(self, url: str, keylog: str | None, cafile: str | None, connect_timeout: float):
        u = urlsplit(url)
        if u.scheme not in ("http", "https") or not u.hostname:
            raise SystemExit(f"bad upstream url: {url}")
        self.host = u.hostname
        self.port = u.port or (443 if u.scheme == "https" else 80)
        default_port = 443 if u.scheme == "https" else 80
        self.host_header = self.host if self.port == default_port else f"{self.host}:{self.port}"
        self.prefix = u.path.rstrip("/")
        self.connect_timeout = connect_timeout
        self.tls = None
        if u.scheme == "https":
            ctx = ssl.create_default_context(cafile=cafile)
            ctx.set_alpn_protocols(["http/1.1"])
            keylog = keylog or os.environ.get("SSLKEYLOGFILE")
            if keylog:
                os.makedirs(os.path.dirname(os.path.abspath(keylog)), exist_ok=True)
                ctx.keylog_filename = keylog
            self.tls = ctx

    async def open(self):
        return await asyncio.wait_for(
            asyncio.open_connection(self.host, self.port, ssl=self.tls,
                                    server_hostname=self.host if self.tls else None, limit=HEAD_LIMIT),
            self.connect_timeout)

    def target(self, target: str) -> str:
        if target.startswith(("http://", "https://")):
            u = urlsplit(target)
            target = (u.path or "/") + (f"?{u.query}" if u.query else "")
        if target == "*":
            return target
        return self.prefix + target


def peer(writer, which: str) -> str | None:
    info = writer.get_extra_info(which)
    return f"{info[0]}:{info[1]}" if info else None


class Gateway:
    def __init__(self, upstream: Upstream, log: Log, idle: float, hold_until: str | None = None,
                 hold_max: float = 0.0):
        self.up = upstream
        self.log = log
        self.idle = idle
        self.next_conn = 0
        self.next_up = 0
        self.tasks: set[asyncio.Task] = set()
        self.hold_until, self.hold_max = hold_until, hold_max
        self.hold_lock = asyncio.Lock()
        self.held = hold_until is None

    async def hold(self, conn: int) -> None:
        if self.held:
            return
        async with self.hold_lock:
            if self.held:
                return
            t0 = time.time()
            while not os.path.exists(self.hold_until) and time.time() - t0 < self.hold_max:
                await asyncio.sleep(0.25)
            self.held = True
            self.log.write(dict(event="hold", conn=conn, path=self.hold_until, waited=round(time.time() - t0, 3),
                                released="file" if os.path.exists(self.hold_until) else "timeout", t=time.time()))

    async def handle(self, d_reader, d_writer):
        self.next_conn += 1
        conn = self.next_conn
        task = asyncio.current_task()
        self.tasks.add(task)
        up = None   # (reader, writer, up_conn_id)
        seq = 0
        try:
            while True:
                try:
                    raw = await d_reader.readuntil(b"\r\n\r\n")
                except asyncio.IncompleteReadError as e:
                    if e.partial.strip():
                        raise
                    return
                except (ConnectionError, asyncio.LimitOverrunError):
                    return
                if not raw.strip():
                    continue
                seq += 1
                up, keep = await self.call(conn, seq, Head(raw), d_reader, d_writer, up)
                if not keep:
                    return
        except Exception as e:
            self.log.write(dict(event="error", conn=conn, t=time.time(), error=f"{type(e).__name__}: {e}"))
        finally:
            if up is not None:
                self.close_up(up, conn, "agent connection closed")
            d_writer.close()
            self.tasks.discard(task)

    def close_up(self, up, conn, reason):
        up[1].close()
        self.log.write(dict(event="up_close", conn=conn, up_conn=up[2], reason=reason, t=time.time()))

    async def call(self, conn, seq, req: Head, d_reader, d_writer, up):
        t0 = time.time()
        method, target, version = req.start.split(" ", 2)
        rec = dict(event="call", conn=conn, seq=seq, t=t0, method=method, path=target, model=None, up_conn=None,
                   up_new=False, up_local=None, up_remote=None, req_framing=None, req_body_bytes=0,
                   req_wire_body_bytes=0, req_head_bytes=req.size(), req_sha256=None,
                   req_content_type=req.get("content-type"), req_content_encoding=req.get("content-encoding"),
                   expect_continue=False, status=None, resp_framing=None, resp_body_bytes=0,
                   resp_wire_body_bytes=0, resp_head_bytes=0, resp_pieces=0,
                   resp_content_type=None, resp_content_encoding=None, upgrade=False,
                   t_req_end=None, t_resp_head=None, t_first_byte=None, t_end=None, error=None, error_side=None)
        client_close = "close" in req.tokens("connection") or (version == "HTTP/1.0"
                                                               and "keep-alive" not in req.tokens("connection"))
        await self.hold(conn)
        if up is not None and (up[0].at_eof() or up[1].is_closing()):
            self.close_up(up, conn, "upstream closed while idle")
            up = None
        try:
            if up is None:
                r, w = await self.up.open()
                self.next_up += 1
                up = (r, w, self.next_up)
                rec["up_new"] = True
                self.log.write(dict(event="up_open", conn=conn, up_conn=self.next_up, t=time.time(),
                                    up_local=peer(w, "sockname"), up_remote=peer(w, "peername")))
        except Exception as e:
            rec["error"] = f"upstream connect failed: {type(e).__name__}: {e}"
            rec["error_side"] = "upstream"
            rec["t_end"] = time.time()
            self.log.write(rec)
            body = json.dumps({"error": rec["error"]}).encode()
            d_writer.write(b"HTTP/1.1 502 Bad Gateway\r\nContent-Type: application/json\r\nContent-Length: "
                           + str(len(body)).encode() + b"\r\nConnection: close\r\n\r\n" + body)
            await d_writer.drain()
            return None, False
        u_reader, u_writer, up_id = up
        rec.update(up_conn=up_id, up_local=peer(u_writer, "sockname"), up_remote=peer(u_writer, "peername"))
        sha = hashlib.sha256()
        scan = bytearray()

        def seen(piece):
            sha.update(piece)
            if len(scan) < MODEL_SCAN:
                scan.extend(piece[:MODEL_SCAN - len(scan)])

        keep_up = True
        sent = got = None
        try:
            try:
                kind, length = framing(req, True)
            except ValueError as e:
                raise SideError("agent", e) from e
            rec["req_framing"] = kind
            drop = set()
            if "100-continue" in req.tokens("expect"):
                # the gateway answers 100 Continue itself so the agent sends its body at once
                rec["expect_continue"] = True
                drop.add("expect")
                await on_side("agent", send(d_writer, b"HTTP/1.1 100 Continue\r\n\r\n"))
            u_writer.write(req.encode(f"{method} {self.up.target(target)} {version}",
                                      replace={"host": self.up.host_header}, drop=drop))
            sent = Relay(self.idle, "agent", "upstream", seen)
            await sent.run(kind, length, d_reader, u_writer)
            await on_side("upstream", u_writer.drain())
            rec.update(req_body_bytes=sent.body, req_wire_body_bytes=sent.wire, t_req_end=time.time())
            if kind != "none":
                rec["req_sha256"] = sha.hexdigest()
                m = MODEL_RE.search(bytes(scan))
                rec["model"] = m.group(1).decode("utf-8", "replace") if m else None

            while True:
                raw = await on_side("upstream", asyncio.wait_for(u_reader.readuntil(b"\r\n\r\n"), self.idle))
                try:
                    resp = Head(raw)
                    status = int(resp.start.split(" ", 2)[1])
                except (ValueError, IndexError) as e:
                    raise SideError("upstream", ValueError(f"bad response head: {e}")) from e
                rec["resp_head_bytes"] += resp.size()
                await on_side("agent", send(d_writer, raw))
                if 100 <= status < 200 and status != 101:
                    continue
                break
            rec.update(status=status, t_resp_head=time.time(), resp_content_type=resp.get("content-type"),
                       resp_content_encoding=resp.get("content-encoding"))

            if status == 101:
                rec["upgrade"] = True
                up_bytes, down_bytes = await self.tunnel(d_reader, d_writer, u_reader, u_writer)
                rec.update(req_wire_body_bytes=rec["req_wire_body_bytes"] + up_bytes,
                           resp_body_bytes=down_bytes, resp_wire_body_bytes=down_bytes, resp_framing="tunnel",
                           t_end=time.time())
                self.log.write(rec)
                self.close_up(up, conn, "tunnel ended")
                return None, False

            r_kind, r_len = framing(resp, False, method, status)
            rec["resp_framing"] = r_kind
            got = Relay(self.idle, "upstream", "agent")
            await got.run(r_kind, r_len, u_reader, d_writer)
            rec.update(resp_body_bytes=got.body, resp_wire_body_bytes=got.wire, resp_pieces=got.pieces,
                       t_first_byte=got.t_first, t_end=time.time())
            if r_kind == "eof" or "close" in resp.tokens("connection"):
                keep_up = False
        except Exception as e:
            side = e.side if isinstance(e, SideError) else "gateway"
            rec.update(error=str(e) if isinstance(e, SideError) else f"{type(e).__name__}: {e}", error_side=side,
                       t_end=time.time())
            if sent is not None and rec["t_req_end"] is None:
                rec.update(req_body_bytes=sent.body, req_wire_body_bytes=sent.wire)
            if got is not None:
                rec.update(resp_body_bytes=got.body, resp_wire_body_bytes=got.wire, resp_pieces=got.pieces,
                           t_first_byte=got.t_first)
            self.log.write(rec)
            self.close_up(up, conn, {"agent": "agent connection failed during the call",
                                     "upstream": "upstream connection failed during the call"}.get(
                                         side, "gateway error during the call"))
            return None, False

        self.log.write(rec)
        if r_kind == "eof":
            self.close_up(up, conn, "response ended by upstream close")
            return None, False
        if client_close:
            self.close_up(up, conn, "agent asked to close")
            return None, False
        if not keep_up:
            self.close_up(up, conn, "upstream asked to close")
            return None, True
        return up, True

    async def tunnel(self, d_reader, d_writer, u_reader, u_writer):
        async def pipe(src, dst):
            n = 0
            while True:
                data = await src.read(PIECE)
                if not data:
                    break
                dst.write(data)
                await dst.drain()
                n += len(data)
            dst.close()
            return n
        a, b = await asyncio.gather(pipe(d_reader, u_writer), pipe(u_reader, d_writer), return_exceptions=True)
        return (a if isinstance(a, int) else 0), (b if isinstance(b, int) else 0)


async def serve(args):
    host, _, port = args.listen.rpartition(":")
    log = Log(args.log)
    gw = Gateway(Upstream(args.upstream, args.keylog, args.cafile, args.connect_timeout), log, args.idle_timeout,
                 args.hold_until, args.hold_max)
    server = await asyncio.start_server(gw.handle, host or "127.0.0.1", int(port), limit=HEAD_LIMIT)
    stop = asyncio.Event()
    loop = asyncio.get_running_loop()
    for sig in (signal.SIGINT, signal.SIGTERM):
        loop.add_signal_handler(sig, stop.set)
    print(f"call_gateway listening on {args.listen} -> {args.upstream}", file=sys.stderr, flush=True)
    async with server:
        await stop.wait()
        server.close()
        for task in list(gw.tasks):
            task.cancel()
        await asyncio.gather(*gw.tasks, return_exceptions=True)
    log.close()


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--listen", default="127.0.0.1:8081", help="host:port the agent calls")
    p.add_argument("--upstream", required=True, help="model endpoint base URL, e.g. https://api.deepseek.com/anthropic")
    p.add_argument("--log", required=True, help="JSON-lines call log")
    p.add_argument("--keylog", default=None, help="TLS key log for ground-truth labelling (default: SSLKEYLOGFILE)")
    p.add_argument("--cafile", default=None, help="CA bundle trusted instead of the system store (tests only)")
    p.add_argument("--connect-timeout", type=float, default=30.0)
    p.add_argument("--idle-timeout", type=float, default=900.0, help="longest silence inside one call")
    p.add_argument("--hold-until", default=None,
                   help="warm-home preparation only: hold requests until this file exists")
    p.add_argument("--hold-max", type=float, default=240.0, help="longest hold in seconds")
    asyncio.run(serve(p.parse_args(argv)))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile harness/vm_gate.py
"""VM gate for the benign collection testbed. Run as root on the VM, from the repository root, with the DeepSeek key
in DEEPSEEK_API_KEY (it is never written to disk):

    sudo -E python3 harness/vm_gate.py

Every check below runs on the VM:
  V0 environment     root, Python, the commands the testbed needs, agent versions, the key present
  V1 offline checks  harness/offline_check.py --agents (gateway, MCP tool log, both agents against a local fake
                     endpoint)
  V2 isolation       two namespaces at once: each capture holds only its own run's traffic; a run cannot reach another
                     run, the VM itself or the cloud metadata address
  V3 Anthropic API   inside a namespace, through the gateway: a streamed request with a tool definition is answered
                     (whether the model called the tool is reported); a long streamed answer reaches the client piece
                     by piece; three calls share one TLS connection on the wire
  V4 key log         the run's TLS key log decrypts its capture with tshark; decrypted request lengths equal the
                     gateway's logged lengths; without the key log nothing decrypts
  V5 Responses API   a streamed request with a function tool on DeepSeek's Responses API, the wire format Codex uses,
                     is answered as an event stream (whether the model called the tool is reported)
  V6 Claude Code     a one-tool task through the gateway, with the MCP tool log and a capture
  V7 Codex           the same task with Codex
  V8 fidelity        each agent runs the same task straight to DeepSeek; TLS connections to the model host, offered
                     ALPN protocols and upload bytes are compared with the gateway run
  H1 warm homes      one warm-up run per agent from an empty home makes the agent's template home (homes.py); the
                     Codex warm-up is held at the gateway until its plugin catalogue sync has finished; each template
                     is checked (sync finished, no API key in any file) and saved; reports the warm-up's bytes per host
Gates: gate 1 (Codex on DeepSeek) = V5 and V7; gate 2 (gateway keeps the traffic shape) = V1, V3 and V8;
testbed = V0, V2, V4, V6 and H1.

Every agent run in V6, V7 and V8 starts from a private copy of its agent's template home: a returning user with a warm
home, not a first launch. V6, V7 and V8 report bytes up and down per remote host for every run (TLS server name, or
the server address when there is none; UDP per server address) and fail a run that downloads more than
BACKGROUND_DOWNLOAD_LIMIT bytes from any host other than the model host.

Upload bytes are measured three ways per TCP connection: raw TCP payload of every captured segment (resent segments
included), TCP payload counted once from the sequence numbers (resent bytes removed), and TLS record bytes (record
header plus record length, each record once). V4 and V8 report the last two and the bytes resent.

Wire checks on a run capture (V2 for both runs, V8 for all four agent runs); a run fails when any of them fails:
  - both veth ends have the uplink's MTU and have segmentation and receive offloads off (checked when the run's
    namespace is created, reported here);
  - no ICMP "fragmentation needed" message in the capture;
  - no captured IP packet larger than the MTU (a larger one is an offload block, not a wire packet);
  - upload bytes resent at most RESENT_TOLERANCE_SEGMENTS full-size segments (MTU minus 40 bytes each) over the whole
    capture; expected 0 on a clean link; download bytes resent by the endpoint are reported, not gated;
  - tcpdump wrote every packet its filter received and dropped none.
Every run directory gets net.json: uplink, MTU, offload states, subnet and capture counts.

Upstream drops: when a call through the gateway ends in an error on the endpoint's side (error_side "upstream" in the
gateway log) and the first FIN or RST on its connection in the capture comes from the endpoint, the endpoint dropped
the connection; this is recorded as an upstream drop, not a gateway fault. A check through the gateway (V3, V5, V6,
V7) that meets an upstream drop is run once more and fails only if the rerun meets one too. A call the agent itself
closed (error_side "agent") is reported but is not a gateway fault either.

Key logs are deleted when the gate ends. Captures and logs stay under data/harness/vm_gate/<time>/ (git-ignored).
"""
from __future__ import annotations

import argparse
import http.client
import json
import os
import platform
import re
import shutil
import socket
import ssl
import subprocess
import sys
import time
import urllib.error
import urllib.request

HERE = os.path.dirname(os.path.abspath(__file__))
REPO = os.path.dirname(HERE)
sys.path.insert(0, HERE)

MODEL_HOST = "api.deepseek.com"
ANTHROPIC_BASE = "https://api.deepseek.com/anthropic"
RESPONSES_BASE = "https://api.deepseek.com"
MODEL = "deepseek-flash"
WORD_COUNT_SCHEMA = {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}
AGENT_TIMEOUT = 300
NEEDED = ["ip", "iptables", "tcpdump", "tshark", "ethtool", "node", "npm", "claude", "codex", "openssl"]
RESENT_TOLERANCE_SEGMENTS = 1
HOMES_ROOT = os.path.join(REPO, "data", "harness", "homes")
BACKGROUND_DOWNLOAD_LIMIT = 1_000_000
WARMUP_HOLD_MAX = 240


# ---------------------------------------------------------------- clients run inside a namespace (internal mode)
def client_anthropic(port: int) -> dict:
    """Three calls on one kept-alive connection to the gateway: tool request, long streamed answer, short call."""
    key = os.environ["DEEPSEEK_API_KEY"]
    conn = http.client.HTTPConnection("127.0.0.1", port, timeout=300)
    headers = {"content-type": "application/json", "x-api-key": key, "anthropic-version": "2023-06-01"}
    out = {}

    def call(name, body, stream):
        t0 = time.time()
        conn.request("POST", "/v1/messages", json.dumps(body), headers)
        resp = conn.getresponse()
        reads, text = [], b""
        while True:
            piece = resp.read1(65536) if stream else resp.read()
            if not piece:
                break
            reads.append((time.time(), len(piece)))
            text += piece
            if not stream:
                break
        out[name] = dict(status=resp.status, t0=t0, reads=reads, bytes=len(text),
                         content_type=resp.getheader("content-type"),
                         head=text[:300].decode("utf-8", "replace") if resp.status != 200 else None)
        return text.decode("utf-8", "replace")

    text = call("tool", dict(model=MODEL, max_tokens=256, stream=True,
                             tools=[dict(name="word_count", description="Count the words in a text.",
                                         input_schema=WORD_COUNT_SCHEMA)],
                             messages=[dict(role="user", content="Use the word_count tool to count the words in: "
                                                                 "alpha beta gamma delta")]), True)
    out["tool"]["tool_use"] = bool(re.search(r'"type"\s*:\s*"tool_use"', text)
                                   and re.search(r'"name"\s*:\s*"word_count"', text))
    call("stream", dict(model=MODEL, max_tokens=3000, stream=True, messages=[dict(
        role="user", content="Write the whole numbers from 1 to 400 in order, separated by single spaces, "
                             "and nothing else.")]), True)
    call("short", dict(model=MODEL, max_tokens=16, messages=[dict(role="user", content="Reply with: ok")]), False)
    conn.close()
    return out


def client_responses(port: int) -> dict:
    key = os.environ["DEEPSEEK_API_KEY"]
    conn = http.client.HTTPConnection("127.0.0.1", port, timeout=300)
    body = dict(model=MODEL, stream=True, input="Use the word_count tool to count the words in: alpha beta gamma delta",
                tools=[dict(type="function", name="word_count", description="Count the words in a text.",
                            parameters=WORD_COUNT_SCHEMA)])
    conn.request("POST", "/responses", json.dumps(body),
                 {"content-type": "application/json", "authorization": f"Bearer {key}"})
    resp = conn.getresponse()
    text = b""
    while True:
        piece = resp.read1(65536)
        if not piece:
            break
        text += piece
    conn.close()
    t = text.decode("utf-8", "replace")
    return dict(status=resp.status, content_type=resp.getheader("content-type"), bytes=len(text),
                function_call=bool(re.search(r'"type"\s*:\s*"function_call"', t)
                                   and re.search(r'"name"\s*:\s*"word_count"', t)),
                events=sorted(set(re.findall(r"event:\s*(\S+)", t))),
                head=t[:300] if resp.status != 200 else None)


def client_fetch(url: str) -> dict:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        with opener.open(url, timeout=30) as r:
            return dict(ok=True, status=r.status)
    except urllib.error.HTTPError as e:
        return dict(ok=True, status=e.code)
    except Exception as e:
        return dict(ok=False, error=f"{type(e).__name__}: {e}")


def client_reach(ip: str, port: int) -> dict:
    s = socket.socket()
    s.settimeout(4)
    try:
        s.connect((ip, port))
        return dict(result="connected")
    except ConnectionRefusedError:
        return dict(result="refused")
    except (socket.timeout, TimeoutError):
        return dict(result="timeout")
    except OSError as e:
        return dict(result=f"error {e.errno}")
    finally:
        s.close()


def client_alpn(host: str) -> dict:
    ctx = ssl.create_default_context()
    ctx.set_alpn_protocols(["h2", "http/1.1"])
    with socket.create_connection((host, 443), timeout=15) as raw, ctx.wrap_socket(raw, server_hostname=host) as s:
        return dict(selected=s.selected_alpn_protocol(), tls=s.version())


# ---------------------------------------------------------------- helpers on the VM side
def sh(args, timeout=60, **kw):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout, **kw)


def in_ns(net, client: str, *args, timeout=400) -> dict:
    r = sh(net.cmd([sys.executable, os.path.abspath(__file__), "--client", client, *map(str, args)]), timeout=timeout)
    try:
        return json.loads(r.stdout.strip().splitlines()[-1])
    except (ValueError, IndexError):
        return dict(error=f"client {client} failed: {r.stderr.strip()[-400:]}")


def tshark_rows(pcap, display_filter, fields, keylog=None):
    args = ["tshark", "-r", pcap, "-n"]
    if keylog:
        args += ["-o", f"tls.keylog_file:{keylog}"]
    args += ["-Y", display_filter, "-T", "fields", "-E", "separator=\t"]
    for f in fields:
        args += ["-e", f]
    r = sh(args, timeout=300)
    return [ln.split("\t") for ln in r.stdout.splitlines() if ln.strip()]


NOT_RESENT = ("!tcp.analysis.retransmission && !tcp.analysis.fast_retransmission "
              "&& !tcp.analysis.spurious_retransmission")


def cols(row, n):
    return (list(row) + [""] * n)[:n]


def truthy(value: str) -> bool:
    return value.strip().lower() in ("1", "true")


def epoch(value: str) -> float:
    """frame.time_epoch as seconds; tshark prints a number, some versions an ISO 8601 time."""
    try:
        return float(value)
    except ValueError:
        import datetime
        v = value.strip().replace("Z", "+00:00")
        if "." in v:
            head, frac = v.split(".", 1)
            n = len(frac) - len(frac.lstrip("0123456789"))
            v = f"{head}.{frac[:min(n, 6)]}{frac[n:]}"
        t = datetime.datetime.fromisoformat(v)
        if t.tzinfo is None:
            t = t.replace(tzinfo=datetime.timezone.utc)
        return t.timestamp()


def upload_measures(segments, records, client_ip, flagged=()) -> dict:
    """Bytes per TCP stream, from tshark rows.

    segments  rows (tcp.stream, ip.src, tcp.seq, tcp.len), one per captured segment with payload
    records   rows (tcp.stream, ip.src, tls.record.length values joined by commas), frames that are not resends
    flagged   rows (tcp.stream, ip.src, tcp.len), segments tshark marks as resent
    Per stream:
      up_raw    client payload of every captured segment, resent segments included
      up_tcp    client payload counted once: highest sequence end minus lowest sequence start
      resent    up_raw minus up_tcp
      up_tls    client TLS record bytes: 5-byte record header plus record length, each record once
      down_raw  endpoint payload of every captured segment
      down_resent  endpoint payload of segments tshark marks as resent (a segment the endpoint lost on the way and
                   sent again is seen once in a client-side capture, so the sequence span cannot count it)
    The capture is taken on the client's side of each connection, so every byte the client sent appears there before
    any resend of it; up_tls and up_tcp therefore count the same bytes and should be equal.
    """
    empty = dict(up_raw=0, up_tcp=0, resent=0, up_tls=0, down_raw=0, down_resent=0)
    out, span = {}, {}
    for row in segments:
        stream, src, seq, length = cols(row, 4)
        if not seq.isdigit() or not length.isdigit():
            continue
        f = out.setdefault(stream, dict(empty))
        n, start = int(length), int(seq)
        if src == client_ip:
            f["up_raw"] += n
            lo, hi = span.get(stream, (start, start + n))
            span[stream] = (min(lo, start), max(hi, start + n))
        else:
            f["down_raw"] += n
    for stream, (lo, hi) in span.items():
        out[stream]["up_tcp"] = hi - lo
        out[stream]["resent"] = out[stream]["up_raw"] - (hi - lo)
    for row in records:
        stream, src, lengths = cols(row, 3)
        if src == client_ip and lengths:
            f = out.setdefault(stream, dict(empty))
            f["up_tls"] += sum(5 + int(x) for x in lengths.split(",") if x.strip().isdigit())
    for row in flagged:
        stream, src, length = cols(row, 3)
        if src != client_ip and length.isdigit():
            out.setdefault(stream, dict(empty))["down_resent"] += int(length)
    return out


def flows(pcap: str, client_ip: str) -> dict:
    """Per TCP stream: server name, offered ALPN list, selected ALPN when visible, and the byte counts of
    upload_measures."""
    out = {}
    for row in tshark_rows(pcap, "tls.handshake.type == 1", ["tcp.stream", "tls.handshake.extensions_server_name",
                                                              "tls.handshake.extensions_alpn_str"]):
        stream, sni, alpn = cols(row, 3)
        out.setdefault(stream, {}).update(sni=sni, offered=alpn)
    for row in tshark_rows(pcap, "tls.handshake.type == 2", ["tcp.stream", "tls.handshake.extensions_alpn_str"]):
        stream, alpn = cols(row, 2)
        if alpn:
            out.setdefault(stream, {})["selected"] = alpn
    segments = tshark_rows(pcap, "tcp.len > 0", ["tcp.stream", "ip.src", "tcp.seq", "tcp.len"])
    records = tshark_rows(pcap, f"tls.record && {NOT_RESENT}", ["tcp.stream", "ip.src", "tls.record.length"])
    flagged = tshark_rows(pcap, f"tcp.len > 0 && !({NOT_RESENT})", ["tcp.stream", "ip.src", "tcp.len"])
    for stream, m in upload_measures(segments, records, client_ip, flagged).items():
        out.setdefault(stream, {}).update(m)
    return out


def first(value: str) -> str:
    """Outer header value of a field that ICMP errors repeat for the quoted inner header."""
    return value.split(",")[0]


def wire_facts(pcap: str, fl: dict, meta: dict) -> dict:
    """Capture-level facts for wire_verdict: ICMP fragmentation-needed messages, largest packet, resent bytes."""
    icmp = tshark_rows(pcap, "icmp.type == 3 && icmp.code == 4", ["ip.src", "icmp.mtu"])
    sizes = [int(first(r[0])) for r in tshark_rows(pcap, "ip", ["ip.len"]) if r and first(r[0]).isdigit()]
    cap = next((c for c in meta.get("captures", []) if c["path"] == pcap), {})
    return dict(mtu=meta.get("mtu"), uplink=meta.get("uplink"), uplink_mtu=meta.get("uplink_mtu"),
                host_if_mtu=meta.get("host_if_mtu"), ns_if_mtu=meta.get("ns_if_mtu"),
                offloads=meta.get("offloads", {}), icmp_frag_needed=len(icmp),
                icmp_next_hop_mtu=sorted({first(cols(r, 2)[1]) for r in icmp}), packets=len(sizes),
                largest_packet=max(sizes, default=0), up_resent=sum(f.get("resent", 0) for f in fl.values()),
                down_resent=sum(f.get("down_resent", 0) for f in fl.values()),
                capture={k: cap.get(k) for k in ("captured", "received", "dropped_kernel", "dropped_interface",
                                                 "complete")})


def wire_verdict(w: dict) -> tuple[bool, list[str]]:
    """Pass or fail of a run capture, with the reasons for a failure."""
    import netns
    mtu = w.get("mtu") or 0
    tolerance = RESENT_TOLERANCE_SEGMENTS * max(mtu - 40, 0)
    fails = []
    if not mtu or not w.get("host_if_mtu") == w.get("ns_if_mtu") == mtu:
        fails.append(f"veth MTU {w.get('host_if_mtu')}/{w.get('ns_if_mtu')} is not the uplink MTU {mtu}")
    if not w.get("offloads") or not all(netns.offloads_off(v) for v in w["offloads"].values()):
        fails.append(f"offloads not all off: {w.get('offloads')}")
    if w["icmp_frag_needed"]:
        fails.append(f"{w['icmp_frag_needed']} ICMP fragmentation-needed messages (next-hop MTU "
                     f"{', '.join(w['icmp_next_hop_mtu'])})")
    if w["largest_packet"] > mtu:
        fails.append(f"captured packet of {w['largest_packet']} B is larger than the MTU")
    if w["up_resent"] > tolerance:
        fails.append(f"{w['up_resent']} upload bytes resent, above the tolerance of {tolerance} B")
    if not w["capture"].get("complete"):
        fails.append(f"capture incomplete {w['capture']}")
    return not fails, fails


def describe_wire(w: dict) -> str:
    import netns
    ok, fails = wire_verdict(w)
    c = w["capture"]
    off = bool(w["offloads"]) and all(netns.offloads_off(v) for v in w["offloads"].values())
    return (f"MTU {w['mtu']} (uplink {w['uplink']} {w['uplink_mtu']}, veth {w['host_if_mtu']}/{w['ns_if_mtu']}), "
            f"offloads off {off}, "
            f"ICMP fragmentation-needed {w['icmp_frag_needed']}, largest packet {w['largest_packet']} B, upload "
            f"resent {w['up_resent']} B, download resent {w['down_resent']} B, tcpdump captured {c.get('captured')} "
            f"of {c.get('received')} (dropped {c.get('dropped_kernel')})"
            + ("" if ok else f" [FAIL: {'; '.join(fails)}]"))


def host_table_from_rows(tcp_rows, udp_rows, client_ip, sni) -> dict:
    """Bytes per remote host.

    tcp_rows  (tcp.stream, ip.src, ip.dst, tcp.srcport, tcp.dstport, tcp.len), one per TCP packet
    udp_rows  (ip.src, ip.dst, udp.srcport, udp.dstport, udp.length), one per UDP datagram
    sni       {tcp.stream: TLS server name}
    A TCP connection is named by its TLS server name, or by the server address when it has none; UDP is named by the
    server address (DNS marked). up and down are transport payload bytes as captured, resends included.
    """
    hosts, server = {}, {}
    for row in tcp_rows:
        stream, src, dst, sport, dport, length = cols(row, 6)
        src, dst = first(src), first(dst)
        if stream not in server:
            server[stream] = f"{dst}:{dport}" if src == client_ip else f"{src}:{sport}"
        name = sni.get(stream) or server[stream]
        h = hosts.setdefault(name, dict(streams=set(), up=0, down=0))
        h["streams"].add(("tcp", stream))
        h["up" if src == client_ip else "down"] += int(first(length) or 0)
    for row in udp_rows:
        src, dst, sport, dport, length = cols(row, 5)
        src, dst = first(src), first(dst)
        ip, port = (dst, dport) if src == client_ip else (src, sport)
        name = f"dns {ip}" if first(port) == "53" else f"udp {ip}:{first(port)}"
        h = hosts.setdefault(name, dict(streams=set(), up=0, down=0))
        h["streams"].add(("udp", ip, first(port)))
        h["up" if src == client_ip else "down"] += max(int(first(length) or 8) - 8, 0)
    return {k: dict(conns=len(v["streams"]), up=v["up"], down=v["down"]) for k, v in hosts.items()}


def host_table(pcap: str, client_ip: str, fl: dict) -> dict:
    tcp = tshark_rows(pcap, "tcp && !icmp", ["tcp.stream", "ip.src", "ip.dst", "tcp.srcport", "tcp.dstport",
                                              "tcp.len"])
    udp = tshark_rows(pcap, "udp && !icmp", ["ip.src", "ip.dst", "udp.srcport", "udp.dstport", "udp.length"])
    return host_table_from_rows(tcp, udp, client_ip, {k: v["sni"] for k, v in fl.items() if v.get("sni")})


def background_downloads(hosts: dict, limit: int = BACKGROUND_DOWNLOAD_LIMIT) -> list[tuple[str, int]]:
    """Hosts other than the model host that sent the run more than limit bytes."""
    return sorted(((h, v["down"]) for h, v in hosts.items() if h != MODEL_HOST and v["down"] > limit),
                  key=lambda x: -x[1])


def describe_hosts(hosts: dict) -> str:
    rows = sorted(hosts.items(), key=lambda kv: -(kv[1]["up"] + kv[1]["down"]))
    return ", ".join(f"{h} ({v['conns']} conn) up {v['up']:,} B down {v['down']:,} B" for h, v in rows) or "none"


def close_rows_to_events(rows, client_ip) -> dict:
    """FIN and RST packets per client port, in capture order: {port: [(time, sent by the endpoint, 'FIN'/'RST')]}.
    rows: (frame.time_epoch, ip.src, tcp.srcport, tcp.dstport, tcp.flags.fin, tcp.flags.reset)."""
    events = {}
    for row in rows:
        t, src, sport, dport, fin, rst = cols(row, 6)
        from_endpoint = src != client_ip
        port = dport if from_endpoint else sport
        events.setdefault(port, []).append((epoch(t), from_endpoint, "RST" if truthy(rst) else "FIN"))
    return events


def close_events(pcap: str, client_ip: str) -> dict:
    rows = tshark_rows(pcap, "tcp.flags.fin == 1 || tcp.flags.reset == 1",
                       ["frame.time_epoch", "ip.src", "tcp.srcport", "tcp.dstport", "tcp.flags.fin",
                        "tcp.flags.reset"])
    return close_rows_to_events(rows, client_ip)


def upstream_drops(calls: list[dict], events: dict) -> list[dict]:
    """Calls through the gateway that ended in an error on the endpoint's side, on a connection the endpoint closed
    first."""
    drops = []
    for c in calls:
        if not c.get("error") or not c.get("up_local") or c.get("error_side", "upstream") != "upstream":
            continue
        port = c["up_local"].rsplit(":", 1)[1]
        after = [e for e in events.get(port, []) if e[0] >= c["t"] - 0.001]
        if not after or not after[0][1]:
            continue
        t, _, kind = after[0]
        if c.get("status") is None:
            phase = "before the response head"
        elif not c.get("resp_body_bytes"):
            phase = f"after the {c['status']} response head, before any body"
        else:
            phase = f"inside the body, after {c['resp_body_bytes']} bytes"
        drops.append(dict(conn=c["conn"], seq=c["seq"], path=c["path"].split("?")[0], status=c.get("status"),
                          phase=phase, closed_by=f"endpoint {kind}",
                          seconds_after_head=round(t - c["t_resp_head"], 3) if c.get("t_resp_head") else None,
                          gateway_error=c["error"]))
    return drops


def describe_drops(drops: list[dict]) -> str:
    return "; ".join(f"upstream drop on {d['path']} {d['phase']} ({d['closed_by']}"
                     + (f", {d['seconds_after_head']} s after the head" if d["seconds_after_head"] is not None
                        else "") + ")" for d in drops)


def model_flows(fl: dict) -> list[dict]:
    return [f for f in fl.values() if f.get("sni") == MODEL_HOST]


def answered(text: str) -> bool:
    """The one-tool task asks for the word count only; the answer must contain the number 4 and no other number."""
    return re.findall(r"\d+", text or "") == ["4"]


def read_jsonl(path, event=None):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        rows = [json.loads(x) for x in f if x.strip()]
    return [r for r in rows if event is None or r.get("event") == event]


class Gate:
    def __init__(self, out_dir, key):
        self.out = out_dir
        self.key = key
        self.results = []
        self.keylogs = []

    def record(self, check, passed, detail, data=None):
        self.results.append(dict(check=check, where="VM", passed=passed, detail=detail, data=data))
        mark = "PASS" if passed else ("SKIP" if passed is None else "FAIL")
        print(f"{mark}  {check}: {detail}", flush=True)

    def start_gateway(self, net, run_dir, name, upstream, port, extra=()):
        keylog = os.path.join(run_dir, f"{name}.keys")
        log = os.path.join(run_dir, f"{name}_calls.jsonl")
        self.keylogs.append(keylog)
        proc = subprocess.Popen(net.cmd([sys.executable, os.path.join(HERE, "call_gateway.py"), "--listen",
                                         f"127.0.0.1:{port}", "--upstream", upstream, "--log", log,
                                         "--keylog", keylog, *extra]), stderr=subprocess.PIPE, text=True)
        line = proc.stderr.readline()
        if "listening" not in line:
            raise RuntimeError(f"gateway did not start: {line}")
        return proc, log, keylog

    @staticmethod
    def stop(proc):
        proc.terminate()
        try:
            proc.wait(10)
        except subprocess.TimeoutExpired:
            proc.kill()

    # V0
    def environment(self):
        versions, missing = {}, []
        for cmd in NEEDED:
            if not shutil.which(cmd):
                missing.append(cmd)
                continue
            flag = {"tshark": ["--version"], "tcpdump": ["--version"], "ip": ["-V"],
                    "openssl": ["version"]}.get(cmd, ["--version"])
            r = sh([cmd] + flag)
            versions[cmd] = (r.stdout or r.stderr).strip().splitlines()[0] if (r.stdout or r.stderr).strip() else "?"
        try:
            import netns
            up = netns.uplink_mtu()
        except Exception as e:
            up = dict(error=f"{type(e).__name__}: {e}")
        ok = os.geteuid() == 0 and sys.version_info >= (3, 9) and not missing and bool(self.key) and "mtu" in up
        self.record("V0 environment", ok,
                    f"root {os.geteuid() == 0}; Python {platform.python_version()}; {platform.platform()}; "
                    f"missing commands {missing or 'none'}; DeepSeek key present {bool(self.key)}; uplink {up}",
                    dict(versions=versions, uplink=up))
        return ok

    # V1
    def offline(self):
        d = os.path.join(self.out, "offline")
        r = sh([sys.executable, os.path.join(HERE, "offline_check.py"), "--agents", "--out", d], timeout=900)
        rep = json.load(open(os.path.join(d, "offline_check.json"))) if os.path.exists(
            os.path.join(d, "offline_check.json")) else []
        failed = [x["check"] for x in rep if x["passed"] is False]
        skipped = [x["check"] for x in rep if x["passed"] is None]
        ok = r.returncode == 0 and rep and not failed and not skipped
        self.record("V1 offline checks", ok, f"{len(rep)} checks; failed {failed or 'none'}; skipped "
                                             f"{skipped or 'none'}", dict(results=rep))

    # V2
    def isolation(self):
        import netns
        d = os.path.join(self.out, "isolation")
        with netns.RunNet(11) as a, netns.RunNet(12) as b:
            ca, cb = a.capture(os.path.join(d, "a.pcap")), b.capture(os.path.join(d, "b.pcap"))
            try:
                me = os.path.abspath(__file__)
                pa = subprocess.Popen(a.cmd([sys.executable, me, "--client", "fetch", f"https://{MODEL_HOST}/"]),
                                      stdout=subprocess.PIPE, text=True)
                pb = subprocess.Popen(b.cmd([sys.executable, me, "--client", "fetch", "https://www.example.com/"]),
                                      stdout=subprocess.PIPE, text=True)
                fa = json.loads((pa.communicate(timeout=60)[0].strip().splitlines() or ["{}"])[-1])
                fb = json.loads((pb.communicate(timeout=60)[0].strip().splitlines() or ["{}"])[-1])
                time.sleep(1)
            finally:
                ca.stop()
                cb.stop()
            meta_a = a.write_metadata(os.path.join(d, "a_net.json"))
            meta_b = b.write_metadata(os.path.join(d, "b_net.json"))
            # reachability is tested after the captures stop, so its probe packets are not part of the capture check
            reach_run = in_ns(a, "reach", b.ns_ip, 9)
            reach_vm = in_ns(a, "reach", a.host_ip, 22)
            reach_meta = in_ns(a, "reach", "169.254.169.254", 80)
            ra = tshark_rows(ca.path, "ip", ["ip.src", "ip.dst"])
            rb = tshark_rows(cb.path, "ip", ["ip.src", "ip.dst"])
            sni_a = {r[0] for r in tshark_rows(ca.path, "tls.handshake.type == 1",
                                               ["tls.handshake.extensions_server_name"])}
            sni_b = {r[0] for r in tshark_rows(cb.path, "tls.handshake.type == 1",
                                               ["tls.handshake.extensions_server_name"])}
            ips_a = {ip for row in ra for cell in row for ip in cell.split(",")}
            ips_b = {ip for row in rb for cell in row for ip in cell.split(",")}
            clean = (b.ns_ip not in ips_a and a.ns_ip not in ips_b and a.ns_ip in ips_a and b.ns_ip in ips_b)
            wa = wire_facts(ca.path, flows(ca.path, a.ns_ip), meta_a)
            wb = wire_facts(cb.path, flows(cb.path, b.ns_ip), meta_b)
            ok = (fa.get("ok") and fb.get("ok") and sni_a == {MODEL_HOST} and sni_b == {"www.example.com"} and clean
                  and reach_run.get("result") == "timeout" and reach_vm.get("result") == "timeout"
                  and reach_meta.get("result") == "timeout" and wire_verdict(wa)[0] and wire_verdict(wb)[0])
            self.record("V2 isolation", ok,
                        f"run A fetched {MODEL_HOST} ({fa}), run B fetched www.example.com ({fb}) at the same time; "
                        f"capture A: {len(ra)} packets, server names {sorted(sni_a)}; capture B: {len(rb)} packets, "
                        f"server names {sorted(sni_b)}; no packet of one run in the other's capture {clean}; "
                        f"A to B {reach_run}, A to the VM {reach_vm}, A to the metadata address {reach_meta}; "
                        f"wire A: {describe_wire(wa)}; wire B: {describe_wire(wb)}", dict(wire_a=wa, wire_b=wb))

    # V3 and V4
    def with_rerun(self, attempt, index, rerun_index, name):
        """Runs one attempt; when the endpoint dropped a call, records the drop and runs the attempt once more."""
        first = attempt(index, name)
        if not first["drops"]:
            return first, ""
        note = f"first attempt: {describe_drops(first['drops'])}; rerun once"
        second = attempt(rerun_index, name + "_rerun")
        second["first_drops"] = first["drops"]
        if second["drops"]:
            note += f"; rerun: {describe_drops(second['drops'])}, so the endpoint dropped a call again"
        else:
            note += "; the rerun had no upstream drop"
        return second, note

    def anthropic_attempt(self, index, name):
        import netns
        d = os.path.join(self.out, name)
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(d, "run.pcap"))
            gw = None
            try:
                gw, log, keylog = self.start_gateway(net, d, "gw", ANTHROPIC_BASE, 8081)
                res = in_ns(net, "anthropic", 8081)
                time.sleep(0.5)
            finally:
                if gw:
                    self.stop(gw)
                time.sleep(1)
                cap.stop()
            meta = net.write_metadata(os.path.join(d, "net.json"))
            calls = read_jsonl(log, "call")
            return dict(res=res, calls=calls, flows=flows(cap.path, net.ns_ip), pcap=cap.path, keylog=keylog, net=meta,
                        drops=upstream_drops(calls, close_events(cap.path, net.ns_ip)))

    def anthropic_api(self):
        a, note = self.with_rerun(self.anthropic_attempt, 13, 23, "anthropic_api")
        lead = note + "; " if note else ""
        res, calls, fl = a["res"], a["calls"], a["flows"]
        data = dict(client=res, calls=calls, flows=fl, first_attempt_drops=a.get("first_drops"),
                    drops=a["drops"])
        if a["drops"]:
            self.record("V3 Anthropic API", False, note, data)
            self.record("V4 key log", False, "not run: the endpoint dropped a call on both attempts")
            return
        mf = model_flows(fl)
        if "error" in res or len(calls) != 3:
            self.record("V3 Anthropic API", False, f"{lead}client result {str(res)[:400]}; {len(calls)} logged calls",
                        data)
            self.record("V4 key log", False, "not run: V3 did not complete")
            return
        tool_ok = res["tool"]["status"] == 200
        s = res["stream"]
        g = calls[1]
        first_arrival = s["reads"][0][0] if s["reads"] else None
        span = (g["t_end"] - g["t_first_byte"]) if g["t_first_byte"] else 0
        lag = (first_arrival - g["t_first_byte"]) if first_arrival and g["t_first_byte"] else None
        streamed = (s["status"] == 200 and len(s["reads"]) >= 5 and lag is not None and lag < 0.25
                    and span >= 1.0 and first_arrival < g["t_end"] - 0.5)
        one_conn = len({c["up_conn"] for c in calls}) == 1 and [c["up_new"] for c in calls] == [True, False, False]
        ok = tool_ok and streamed and one_conn and len(mf) == 1 and all(c["status"] == 200 for c in calls)
        self.record("V3 Anthropic API", ok,
                    f"{lead}tool request {res['tool']['status']}, tool call returned {res['tool']['tool_use']}; long "
                    f"answer streamed over {span:.2f} s in {len(s['reads'])} reads, first piece reached the client "
                    f"{(lag or 0) * 1e3:.0f} ms after the gateway read it; statuses "
                    f"{[c['status'] for c in calls]}; one upstream connection for 3 calls {one_conn}; TLS "
                    f"connections to {MODEL_HOST} in the capture {len(mf)}"
                    + (f"; error body {res['tool']['head'] or s['head']}" if not tool_ok or s['status'] != 200
                       else ""), data)
        # V4: decrypt with the run's key log, then without it
        req = tshark_rows(a["pcap"], "http.request", ["http.request.method", "http.request.uri",
                                                      "http.content_length"], keylog=a["keylog"])
        rsp = tshark_rows(a["pcap"], "http.response", ["http.response.code"], keylog=a["keylog"])
        plain = tshark_rows(a["pcap"], "http", ["frame.number"])
        dec_lengths = [int(r[2]) for r in req if len(r) > 2 and r[2].isdigit()]
        logged = [c["req_body_bytes"] for c in calls]
        f = mf[0] if mf else {}
        framed = sum(c["req_head_bytes"] + c["req_wire_body_bytes"] for c in calls)
        ok4 = dec_lengths == logged and [r[0] for r in rsp] == ["200"] * 3 and not plain
        self.record("V4 key log", ok4,
                    f"with the key log tshark decrypted {len(req)} requests with lengths {dec_lengths} (gateway "
                    f"logged {logged}) and {len(rsp)} responses {[r[0] for r in rsp]}; without it "
                    f"{len(plain)} HTTP frames; upload on the wire {f.get('up_tls', 0)} bytes of TLS records and "
                    f"{f.get('up_tcp', 0)} TCP bytes without resends ({f.get('resent', 0)} bytes resent) for "
                    f"{framed} bytes of HTTP, so TLS overhead {f.get('up_tls', 0) - framed} bytes over 3 calls")

    # V5
    def responses_attempt(self, index, name):
        import netns
        d = os.path.join(self.out, name)
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(d, "run.pcap"))
            gw = None
            try:
                gw, log, _ = self.start_gateway(net, d, "gw", RESPONSES_BASE, 8082)
                res = in_ns(net, "responses", 8082)
                alpn = in_ns(net, "alpn", MODEL_HOST)
            finally:
                if gw:
                    self.stop(gw)
                time.sleep(1)
                cap.stop()
            meta = net.write_metadata(os.path.join(d, "net.json"))
            calls = read_jsonl(log, "call")
            return dict(res=res, alpn=alpn, calls=calls, pcap=cap.path, net=meta,
                        drops=upstream_drops(calls, close_events(cap.path, net.ns_ip)))

    def responses_api(self):
        a, note = self.with_rerun(self.responses_attempt, 14, 24, "responses_api")
        self.alpn = a["alpn"]
        res, calls = a["res"], a["calls"]
        data = dict(client=res, calls=calls, first_attempt_drops=a.get("first_drops"), drops=a["drops"])
        if a["drops"]:
            self.record("V5 Responses API", False, note, data)
            return
        ok = res.get("status") == 200 and len(calls) == 1 \
            and (calls[0]["resp_content_type"] or "").startswith("text/event-stream")
        self.record("V5 Responses API", ok,
                    (note + "; " if note else "")
                    + f"POST /responses {res.get('status')}, content type {res.get('content_type')}, function call "
                    f"to word_count {res.get('function_call')}, events {res.get('events')}"
                    + (f"; error body {res.get('head') or res.get('error')}" if not ok else ""), data)

    # V6 to V8
    def agent_run(self, which, mode, index, template):
        """Gateway runs get the upstream-drop rerun; direct runs have no gateway log to read drops from."""
        if mode == "gateway":
            return self.with_rerun(lambda i, name: self.agent(which, mode, i, name, template), index, index + 10,
                                   f"{which}_{mode}")
        return self.agent(which, mode, index, f"{which}_{mode}", template), ""

    def agent(self, which, mode, index, name, template=None, hold=False):
        """One run of the one-tool task. template: the template home copied into the run's home before the agent
        starts (None for a warm-up from an empty home). hold: warm-up only, hold the first model call at the gateway
        until the agent's startup work has finished (homes.hold_file)."""
        import homes
        import netns
        from offline_check import AGENT_PROMPT, MCP, codex_config
        d = os.path.join(self.out, name)
        home, ws = os.path.join(d, "home"), os.path.join(d, "ws")
        if template:
            homes.copy_template(template, home)
        os.makedirs(home, exist_ok=True)
        os.makedirs(ws, exist_ok=True)
        hold_file = homes.hold_file(which, home) if hold else None
        hold_args = ["--hold-until", hold_file, "--hold-max", str(WARMUP_HOLD_MAX)] if hold_file else []
        tool_log = os.path.join(d, "tools.jsonl")
        mcp = {"command": sys.executable, "args": [MCP], "env": {"UT_TOOL_LOG": tool_log, "UT_WORKSPACE": ws}}
        env = {k: v for k, v in os.environ.items() if not k.startswith(("ANTHROPIC_", "OPENAI_", "CODEX_", "CLAUDE_"))
               and k.lower() not in ("http_proxy", "https_proxy", "all_proxy")}
        env.update(HOME=home, UT_TOOL_LOG=tool_log, UT_WORKSPACE=ws)
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(d, "run.pcap"))
            gw = log = None
            if which == "claude":
                if mode == "gateway":
                    gw, log, _ = self.start_gateway(net, d, "gw", ANTHROPIC_BASE, 8081, hold_args)
                base = "http://127.0.0.1:8081" if mode == "gateway" else ANTHROPIC_BASE
                cfg = os.path.join(d, "mcp.json")
                with open(cfg, "w") as f:
                    json.dump({"mcpServers": {"bench": dict(mcp, type="stdio")}}, f)
                env.update(ANTHROPIC_BASE_URL=base, ANTHROPIC_AUTH_TOKEN=self.key,
                           ANTHROPIC_MODEL="deepseek-flash[1m]", ANTHROPIC_DEFAULT_OPUS_MODEL="deepseek-flash[1m]",
                           ANTHROPIC_DEFAULT_SONNET_MODEL="deepseek-flash[1m]",
                           ANTHROPIC_DEFAULT_HAIKU_MODEL="deepseek-flash")
                cmd = ["claude", "-p", AGENT_PROMPT, "--mcp-config", cfg, "--strict-mcp-config", "--allowedTools",
                       "mcp__bench__word_count", "--output-format", "json", "--max-turns", "6"]
            else:
                if mode == "gateway":
                    gw, log, _ = self.start_gateway(net, d, "gw", RESPONSES_BASE, 8082, hold_args)
                base = "http://127.0.0.1:8082" if mode == "gateway" else RESPONSES_BASE + "/"
                codex_home = os.path.join(home, ".codex")
                os.makedirs(codex_home, exist_ok=True)
                with open(os.path.join(codex_home, "config.toml"), "w") as f:
                    f.write(codex_config(base, mcp))
                env.update(CODEX_HOME=codex_home, DEEPSEEK_API_KEY=self.key)
                cmd = ["codex", "exec", "--skip-git-repo-check", "--ephemeral", "-C", ws, "-o",
                       os.path.join(d, "last_message.txt"), AGENT_PROMPT]
            t0 = time.time()
            try:
                p = subprocess.run(net.cmd(cmd), env=env, cwd=ws, stdin=subprocess.DEVNULL, capture_output=True,
                                   text=True, timeout=AGENT_TIMEOUT)
                code, out, err = p.returncode, p.stdout, p.stderr
            except subprocess.TimeoutExpired as e:
                code, out, err = "timeout", str(e.stdout or ""), str(e.stderr or "")
            finally:
                time.sleep(1)
                if gw:
                    self.stop(gw)
                cap.stop()
            took = time.time() - t0
            with open(os.path.join(d, "agent_stdout.txt"), "w") as f:
                f.write(out)
            with open(os.path.join(d, "agent_stderr.txt"), "w") as f:
                f.write(err)
            meta = net.write_metadata(os.path.join(d, "net.json"))
            fl = flows(cap.path, net.ns_ip)
            events = close_events(cap.path, net.ns_ip)
            wire = wire_facts(cap.path, fl, meta)
            hosts = host_table(cap.path, net.ns_ip, fl)
        answer = ""
        if which == "claude":
            try:
                answer = str(json.loads(out).get("result", ""))
            except ValueError:
                pass
        elif os.path.exists(os.path.join(d, "last_message.txt")):
            with open(os.path.join(d, "last_message.txt")) as f:
                answer = f.read()
        calls = read_jsonl(log, "call") if log else []
        tools = read_jsonl(tool_log, "tool_call")
        mf = model_flows(fl)
        return dict(which=which, mode=mode, exit=code, seconds=round(took, 1), answer=answer.strip()[-200:],
                    calls=calls, tools=tools, model_flows=mf, pcap=cap.path, drops=upstream_drops(calls, events),
                    wire=wire, hosts=hosts, home=home, template=template,
                    holds=read_jsonl(log, "hold") if log else [],
                    other_hosts=sorted({f.get("sni") for f in fl.values()
                                        if f.get("sni") and f.get("sni") != MODEL_HOST}),
                    stderr_tail=err.strip()[-400:])

    def agent_check(self, name, r, note=""):
        if r["drops"]:
            self.record(name, False, note, r)
            return
        model_calls = [c for c in r["calls"] if c["path"].split("?")[0].endswith(("/v1/messages", "/responses"))]
        streamed = [c for c in model_calls if (c["resp_content_type"] or "").startswith("text/event-stream")]
        agent_conns = {c["conn"] for c in r["calls"]}
        up_conns = {c["up_conn"] for c in r["calls"] if c["up_conn"]}
        agent_closes = [c for c in r["calls"] if c["error"] and c.get("error_side") == "agent"]
        errors = [c["error"] for c in r["calls"] if c["error"] and c.get("error_side") != "agent"]
        background = background_downloads(r["hosts"])
        ok = (r["exit"] == 0 and answered(r["answer"]) and len(model_calls) >= 2 and streamed and r["template"]
              and not background
              and all(c["status"] == 200 for c in model_calls) and r["tools"] and r["tools"][0]["tool"] == "word_count"
              and len(agent_conns) == len(up_conns) and not errors and r["model_flows"])
        self.record(name, ok,
                    (note + "; " if note else "")
                    + f"exit {r['exit']} in {r['seconds']} s, answer {r['answer'][-40:]!r}; {len(model_calls)} model "
                    f"calls (statuses {[c['status'] for c in model_calls]}, {len(streamed)} streamed, request bytes "
                    f"{[c['req_body_bytes'] for c in model_calls]}); tool calls "
                    f"{[(t['tool'], t['args_bytes']) for t in r['tools']]}; "
                    f"agent connections {len(agent_conns)}, upstream connections {len(up_conns)}; calls the agent "
                    f"closed itself {len(agent_closes)}; gateway or endpoint errors {errors}; TLS connections "
                    f"to {MODEL_HOST} {len(r['model_flows'])}; warm home {bool(r['template'])}; bytes per host: "
                    f"{describe_hosts(r['hosts'])}"
                    + (f"; background downloads over {BACKGROUND_DOWNLOAD_LIMIT:,} B: {background}" if background
                       else "")
                    + (f"; stderr {r['stderr_tail'][-300:]}" if not ok else ""), r)

    def fidelity(self, direct, gateway, label):
        def summary(r):
            mf = r["model_flows"]
            total = lambda k: sum(f.get(k, 0) for f in mf)
            return dict(connections=len(mf), offered=sorted({f.get("offered", "") for f in mf}),
                        selected=sorted({f.get("selected", "") for f in mf if f.get("selected")}),
                        up_tls=total("up_tls"), up_tcp=total("up_tcp"), up_raw=total("up_raw"),
                        resent=total("resent"), tool_calls=len(r["tools"]), exit=r["exit"],
                        answer=r["answer"][-20:])

        def pct(g, d):
            return f"{(g - d) / d * 100:+.1f}%" if d else "n/a"
        sd, sg = summary(direct), summary(gateway)
        wire_ok = wire_verdict(direct["wire"])[0] and wire_verdict(gateway["wire"])[0]
        bg_direct, bg_gateway = background_downloads(direct["hosts"]), background_downloads(gateway["hosts"])
        warm = bool(direct["template"]) and bool(gateway["template"])
        probe = getattr(self, "alpn", None) or {}
        server_h2 = probe.get("selected") == "h2"
        direct_h2 = any("h2" in o.split(",") for o in sd["offered"]) and server_h2
        same_protocol = not direct_h2   # the gateway speaks HTTP/1.1 upstream
        same_conns = sd["connections"] == sg["connections"]
        ok = (direct["exit"] == 0 and gateway["exit"] == 0 and "selected" in probe and same_protocol and same_conns
              and sd["connections"] > 0 and wire_ok and warm and not bg_direct and not bg_gateway)
        self.record(f"V8 fidelity {label}", ok,
                    (f"ALPN probe of {MODEL_HOST} failed {probe}; " if "selected" not in probe else "")
                    + ("the gateway run is the rerun after an upstream drop; " if gateway.get("first_drops") else "")
                    + f"TLS connections to {MODEL_HOST}: straight to DeepSeek {sd['connections']}, through the "
                    f"gateway {sg['connections']}; offered ALPN: straight {sd['offered']}, gateway {sg['offered']}; "
                    f"upload as TLS records: straight {sd['up_tls']} B, gateway {sg['up_tls']} B "
                    f"({pct(sg['up_tls'], sd['up_tls'])}); upload as TCP bytes without resends: straight "
                    f"{sd['up_tcp']} B, gateway {sg['up_tcp']} B ({pct(sg['up_tcp'], sd['up_tcp'])}); resent: "
                    f"straight {sd['resent']} B, gateway {sg['resent']} B; tool calls: straight {sd['tool_calls']}, "
                    f"gateway {sg['tool_calls']}; {MODEL_HOST} picks h2 when offered: {server_h2}; same HTTP "
                    f"version on the wire {same_protocol}; same number of connections {same_conns}; wire straight: "
                    f"{describe_wire(direct['wire'])}; wire through the gateway: {describe_wire(gateway['wire'])}; "
                    f"warm homes {warm}; bytes per host straight: {describe_hosts(direct['hosts'])}; bytes per host "
                    f"through the gateway: {describe_hosts(gateway['hosts'])}"
                    + (f"; background downloads over {BACKGROUND_DOWNLOAD_LIMIT:,} B: straight {bg_direct}, gateway "
                       f"{bg_gateway}" if bg_direct or bg_gateway else ""),
                    dict(direct=sd, gateway=sg, wire_direct=direct["wire"], wire_gateway=gateway["wire"],
                         hosts_direct=direct["hosts"], hosts_gateway=gateway["hosts"]))

    def warm_homes(self) -> dict:
        """H1: one warm-up run per agent from an empty home, saved as that agent version's template home."""
        import homes
        made, lines, data = {}, [], {}
        for i, which in enumerate(("claude", "codex")):
            version = homes.agent_version(which)
            r, note = self.with_rerun(lambda idx, name: self.agent(which, "gateway", idx, name, None, hold=True),
                                      31 + i, 41 + i, f"warmup_{which}")
            hold = r["holds"][0] if r["holds"] else None
            line = (f"{which} {version}: warm-up exit {r['exit']} in {r['seconds']} s, answer {r['answer'][-20:]!r}"
                    + (f", first model call held {hold['waited']} s, released by {hold['released']}" if hold else "")
                    + (f" ({note})" if note else "") + f"; bytes per host: {describe_hosts(r['hosts'])}")
            try:
                if r["exit"] != 0 or not answered(r["answer"]) or r["drops"]:
                    raise RuntimeError("the warm-up run did not complete")
                dest = homes.save_template(r["home"], HOMES_ROOT, which, version,
                                           dict(warmup_run=os.path.relpath(os.path.dirname(r["home"]), self.out),
                                                warmup_hosts=r["hosts"], warmup_hold=hold), self.key)
                t = homes.load_template(HOMES_ROOT, which, version)
                made[which] = t["home"]
                line += f"; template {os.path.relpath(dest, REPO)} ({t['home_bytes']:,} B)" + (
                    f", Codex plugin catalogue {t['codex_plugins']['sha'][:12]} "
                    f"({t['codex_plugins']['repo_bytes']:,} B)" if which == "codex" else "")
                data[which] = dict(template=t, warmup_hosts=r["hosts"], hold=hold)
            except Exception as e:
                line += f"; NOT SAVED: {e}"
                data[which] = dict(error=str(e), warmup_hosts=r["hosts"], hold=hold)
            lines.append(line)
        self.record("H1 warm homes", len(made) == 2, " | ".join(lines), data)
        return made

    def guard(self, name, fn, *args):
        try:
            return fn(*args)
        except Exception as e:
            self.record(name, False, f"stopped with {type(e).__name__}: {e}")
            return None

    def delete_keylogs(self):
        removed = 0
        for root, _, files in os.walk(self.out):
            for f in files:
                if f.endswith(".keys"):
                    os.remove(os.path.join(root, f))
                    removed += 1
        return removed


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--client", default=None, help=argparse.SUPPRESS)
    ap.add_argument("args", nargs="*", help=argparse.SUPPRESS)
    a = ap.parse_args(argv)
    if a.client:
        fn = dict(anthropic=lambda: client_anthropic(int(a.args[0])),
                  responses=lambda: client_responses(int(a.args[0])),
                  fetch=lambda: client_fetch(a.args[0]), reach=lambda: client_reach(a.args[0], int(a.args[1])),
                  alpn=lambda: client_alpn(a.args[0]))[a.client]
        try:
            print(json.dumps(fn()))
        except Exception as e:
            print(json.dumps(dict(error=f"{type(e).__name__}: {e}")))
        return 0

    key = os.environ.get("DEEPSEEK_API_KEY", "")
    if not key and sys.stdin.isatty():
        import getpass
        key = getpass.getpass("DeepSeek API key: ")
        os.environ["DEEPSEEK_API_KEY"] = key
    stamp = time.strftime("%Y%m%d_%H%M%S")
    out = os.path.join(REPO, "data", "harness", "vm_gate", stamp)
    os.makedirs(out, exist_ok=True)
    g = Gate(out, key)
    print(f"VM gate {stamp}, output {out}\n", flush=True)
    try:
        if not g.environment():
            print("\nV0 failed; the other checks need it")
            return 1
        import netns
        netns.cleanup_all()
        g.guard("V1 offline checks", g.offline)
        g.guard("V2 isolation", g.isolation)
        g.guard("V3 Anthropic API", g.anthropic_api)
        g.guard("V5 Responses API", g.responses_api)
        templates = g.guard("H1 warm homes", g.warm_homes) or {}
        runs = {}
        for i, (which, mode) in enumerate([("claude", "gateway"), ("codex", "gateway"), ("claude", "direct"),
                                           ("codex", "direct")]):
            if which not in templates:
                runs[(which, mode)] = (None, "")
                continue
            runs[(which, mode)] = g.guard(f"{which} {mode} run", g.agent_run, which, mode, 15 + i,
                                          templates[which]) or (None, "")
        for name, run_key in (("V6 Claude Code", ("claude", "gateway")), ("V7 Codex", ("codex", "gateway"))):
            run, note = runs[run_key]
            if run:
                g.agent_check(name, run, note)
            else:
                g.record(name, False, "the run did not complete or there is no warm home (H1)")
        for label, which in (("Claude Code", "claude"), ("Codex", "codex")):
            direct, gateway = runs[(which, "direct")][0], runs[(which, "gateway")][0]
            if direct and gateway and not gateway["drops"]:
                g.fidelity(direct, gateway, label)
            else:
                g.record(f"V8 fidelity {label}", False, "a run did not complete or the endpoint dropped a call on "
                                                         "both gateway attempts")
    finally:
        removed = g.delete_keylogs()
        try:
            import netns
            netns.cleanup_all()
        except Exception:
            pass
        print(f"\nkey logs deleted: {removed}")
    passed = {r["check"].split()[0]: r["passed"] for r in g.results}
    fid = [r["passed"] for r in g.results if r["check"].startswith("V8")]
    gates = {
        "GATE 1 Codex on DeepSeek (V5, V7)": all(passed.get(k) for k in ("V5", "V7")),
        "GATE 2 gateway keeps the traffic shape (V1, V3, V8)": all(passed.get(k) for k in ("V1", "V3")) and fid
        and all(fid),
        "TESTBED isolation, capture, key log, Claude Code, warm homes (V0, V2, V4, V6, H1)": all(
            passed.get(k) for k in ("V0", "V2", "V4", "V6", "H1")),
    }
    print()
    for name, ok in gates.items():
        print(f"{'PASS' if ok else 'FAIL'}  {name}")
    with open(os.path.join(out, "vm_gate.json"), "w") as f:
        json.dump(dict(stamp=stamp, gates=gates, results=g.results), f, indent=1, default=str)
    print(f"\nreport {os.path.join(out, 'vm_gate.json')}")
    return 0 if all(gates.values()) else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile harness/offline_check.py
"""Offline checks of the call gateway and the MCP tool log. No network, no root, standard library only.

A local fake endpoint stands in for the model endpoint. The gateway under test runs as its own process, exactly as on
the VM. Checks:
  O1 request sizes      exact request body length logged and forwarded byte for byte, 0 B to 1 MB, fixed-length and
                        chunked uploads
  O2 response sizes     exact response body length logged and relayed byte for byte, fixed-length and chunked
  O3 streaming          streamed events reach the client when the endpoint sends them, not when the response ends
  O4 connection reuse   one upstream connection per client connection, reused across calls; concurrent clients get
                        one each
  O5 reconnect          an endpoint that answers with Connection: close is followed by one new upstream connection,
                        logged as such
  O6 idle close         an endpoint that drops the idle connection between calls is followed by one new upstream
                        connection, and the client's next call succeeds on its kept connection
  O7 client close       a client that closes ends its upstream connection
  O8 no-body responses  HEAD and 204 keep the connection in step
  O9 expect-continue    a client that waits for 100 Continue sends its body and is sized exactly
  O10 TLS key log       an HTTPS upstream works and its session keys reach the key log (needs the openssl command)
  O11 MCP tool log      tool calls are answered and their argument sizes logged exactly
  O12 endpoint drop     an endpoint that drops a stream inside the body: the log keeps the bytes relayed before the
                        drop and names the upstream side
  O13 agent close       an agent that closes after the last event, before the end of the stream: the log keeps the
                        bytes relayed and names the agent side
  O14 warm-up hold      with --hold-until, the first request waits until the file exists (or the hold times out) and
                        is then forwarded and sized exactly; later requests are not held
With --agents, real agent clients run against the fake endpoint through the gateway, with the MCP tool server:
  A1 Claude Code (needs the claude command), A2 Codex (needs the codex command).

Usage: python3 harness/offline_check.py [--agents] [--out DIR]
"""
from __future__ import annotations

import argparse
import asyncio
import hashlib
import json
import os
import random
import re
import shutil
import socket
import ssl
import subprocess
import sys
import tempfile
import threading
import time
from urllib.parse import parse_qs, urlsplit

HERE = os.path.dirname(os.path.abspath(__file__))
GATEWAY = os.path.join(HERE, "call_gateway.py")
MCP = os.path.join(HERE, "mcp_tools.py")
SIZES = [0, 1, 2, 100, 1000, 65535, 65536, 65537, 100_000, 999_999, 1_000_000]
CHUNKED_SIZES = [0, 1, 70_000, 1_000_000]
AGENT_PROMPT = ("Use the word_count tool to count the words in this text: 'alpha beta gamma delta'. "
                "Then reply with only the number.")


def free_port() -> int:
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port


# ---------------------------------------------------------------- fake endpoint
class FakeEndpoint:
    """Minimal HTTP/1.1 endpoint with keep-alive, chunked and fixed-length bodies, SSE streams, and fake Anthropic
    Messages and Responses APIs that call one tool and then answer with its result."""

    def __init__(self, tls_ctx=None, host="127.0.0.1", port=None):
        self.tls_ctx = tls_ctx
        self.host = host
        self.port = port or free_port()
        self.conns = []          # one dict per accepted connection
        self.requests = []       # one dict per request
        self.sse_sent = {}       # stream id -> list of send times
        self.loop = asyncio.new_event_loop()
        self.ready = threading.Event()
        threading.Thread(target=self._run, daemon=True).start()
        self.ready.wait(10)

    def _run(self):
        asyncio.set_event_loop(self.loop)
        server = self.loop.run_until_complete(asyncio.start_server(self.handle, self.host, self.port,
                                                                   ssl=self.tls_ctx, limit=1 << 20))
        self.ready.set()
        self.loop.run_forever()
        server.close()

    async def read_body(self, reader, headers):
        if "chunked" in headers.get("transfer-encoding", "").lower():
            body = bytearray()
            while True:
                size = int((await reader.readuntil(b"\r\n")).split(b";")[0], 16)
                if size == 0:
                    while (await reader.readuntil(b"\r\n")) != b"\r\n":
                        pass
                    return bytes(body)
                body += await reader.readexactly(size)
                await reader.readexactly(2)
        n = int(headers.get("content-length", "0") or 0)
        return await reader.readexactly(n) if n else b""

    async def handle(self, reader, writer):
        conn = dict(id=len(self.conns) + 1, peer=writer.get_extra_info("peername")[1], requests=0, eof=False)
        self.conns.append(conn)
        try:
            while True:
                try:
                    raw = await reader.readuntil(b"\r\n\r\n")
                except asyncio.IncompleteReadError:
                    conn["eof"] = True
                    return
                lines = raw.decode("latin-1").split("\r\n")
                method, target, _ = lines[0].split(" ", 2)
                headers = {}
                for ln in lines[1:]:
                    if ln:
                        k, _, v = ln.partition(":")
                        headers[k.strip().lower()] = v.strip()
                if headers.get("expect", "").lower() == "100-continue":
                    writer.write(b"HTTP/1.1 100 Continue\r\n\r\n")
                body = await self.read_body(reader, headers)
                conn["requests"] += 1
                self.requests.append(dict(conn=conn["id"], method=method, target=target, headers=headers,
                                          length=len(body), sha256=hashlib.sha256(body).hexdigest(), t=time.time()))
                close = await self.route(method, target, headers, body, writer)
                if close == "idle":
                    await asyncio.sleep(0.2)   # the endpoint drops the kept connection while it is idle
                    writer.close()
                    return
                if close:
                    writer.close()
                    return
        except (ConnectionError, asyncio.IncompleteReadError):
            conn["eof"] = True
        finally:
            writer.close()

    @staticmethod
    def head(status, reason, headers):
        out = f"HTTP/1.1 {status} {reason}\r\n" + "".join(f"{k}: {v}\r\n" for k, v in headers) + "\r\n"
        return out.encode("latin-1")

    async def send_fixed(self, writer, status, body, ctype="application/json", extra=(), method="POST"):
        writer.write(self.head(status, "OK" if status < 400 else "Error",
                               [("Content-Type", ctype), ("Content-Length", str(len(body))), *extra])
                     + (b"" if method == "HEAD" else body))
        await writer.drain()

    async def send_chunked(self, writer, pieces, ctype, gap=0.0, stream_id=None, end_gap=0.0, cut_after=None):
        writer.write(self.head(200, "OK", [("Content-Type", ctype), ("Transfer-Encoding", "chunked")]))
        await writer.drain()
        times = self.sse_sent.setdefault(stream_id, []) if stream_id else None
        for i, piece in enumerate(pieces):
            if gap and i:
                await asyncio.sleep(gap)
            if cut_after is not None and i == cut_after:
                writer.close()    # the endpoint drops the connection inside the body
                raise ConnectionError("endpoint dropped the stream")
            if times is not None:
                times.append(time.time())
            writer.write(f"{len(piece):x}\r\n".encode() + piece + b"\r\n")
            await writer.drain()
        if end_gap:
            await asyncio.sleep(end_gap)
        writer.write(b"0\r\n\r\n")
        await writer.drain()

    async def route(self, method, target, headers, body, writer):
        u = urlsplit(target)
        q = {k: v[0] for k, v in parse_qs(u.query).items()}
        path = u.path
        if path.endswith("/size"):
            await self.send_fixed(writer, 200, json.dumps({"len": len(body),
                                                           "sha256": hashlib.sha256(body).hexdigest()}).encode())
        elif path.endswith("/bigresp"):
            data = random.Random(int(q.get("n", 0))).randbytes(int(q.get("n", 0)))
            if q.get("chunked") == "1":
                step = 7919
                await self.send_chunked(writer, [data[i:i + step] for i in range(0, len(data), step)] or [],
                                        "application/octet-stream")
            else:
                await self.send_fixed(writer, 200, data, "application/octet-stream")
        elif path.endswith("/sse"):
            n, gap = int(q.get("events", 5)), float(q.get("gap", 0.25))
            pieces = [f"event: tick\ndata: {{\"i\": {i}}}\n\n".encode() for i in range(n)]
            cut = int(q["cut"]) if "cut" in q else None
            await self.send_chunked(writer, pieces, "text/event-stream", gap, q.get("id", "sse"),
                                    float(q.get("endgap", 0)), cut)
            if cut is not None:
                return True
        elif path.endswith("/close"):
            body = b'{"closing": true}'
            writer.write(self.head(200, "OK", [("Content-Type", "application/json"),
                                               ("Content-Length", str(len(body))), ("Connection", "close")]) + body)
            await writer.drain()
            return True
        elif path.endswith("/idleclose"):
            await self.send_fixed(writer, 200, b'{"idle_close": true}')
            return "idle"
        elif path.endswith("/nocontent"):
            writer.write(self.head(204, "No Content", []))
            await writer.drain()
        elif path.endswith("/head") and method == "HEAD":
            writer.write(self.head(200, "OK", [("Content-Type", "text/plain"), ("Content-Length", "12345")]))
            await writer.drain()
        elif path.endswith("/v1/messages/count_tokens"):
            await self.send_fixed(writer, 200, b'{"input_tokens": 10}')
        elif path.endswith("/v1/messages"):
            await self.anthropic(json.loads(body or b"{}"), writer)
        elif path.endswith("/responses"):
            await self.responses(json.loads(body or b"{}"), writer)
        elif path.endswith("/models"):
            await self.send_fixed(writer, 200, json.dumps({"object": "list", "data": [
                {"id": "deepseek-flash", "object": "model", "created": 0, "owned_by": "deepseek"},
                {"id": "deepseek-v4-pro", "object": "model", "created": 0, "owned_by": "deepseek"}]}).encode())
        else:
            await self.send_fixed(writer, 404, json.dumps({"error": f"no route {method} {path}"}).encode(),
                                  method=method)
        return False

    # fake model logic shared by both APIs: call the word_count tool once, then answer with its result
    @staticmethod
    def find_tool(tools, parent=None):
        for t in tools or []:
            if not isinstance(t, dict):
                continue
            name = str(t.get("name", ""))
            if name.endswith("word_count"):
                return name, (parent or t.get("namespace"))
            inner = t.get("tools")
            if isinstance(inner, list):
                found = FakeEndpoint.find_tool(inner, name or parent)
                if found:
                    return found
        return None

    @staticmethod
    def answer_from(result):
        # the agent wraps the tool output in its own text; the fake model answers with the last number in it
        if result is None:
            return "ok"
        numbers = re.findall(r"\d+", result)
        return numbers[-1] if numbers else result.strip()

    @staticmethod
    def quoted(text):
        m = re.search(r"'([^']+)'", text or "")
        return m.group(1) if m else ""

    async def anthropic(self, req, writer):
        result, prompt = None, ""
        for m in req.get("messages") or []:
            content = m.get("content")
            if isinstance(content, str):
                prompt = content if m.get("role") == "user" else prompt
                continue
            for b in content or []:
                if b.get("type") == "tool_result":
                    c = b.get("content")
                    result = c if isinstance(c, str) else "".join(x.get("text", "") for x in c or [])
                elif b.get("type") == "text" and m.get("role") == "user":
                    prompt = b.get("text", "")
        tool = self.find_tool(req.get("tools"))
        model = req.get("model", "deepseek-flash")
        forced = (req.get("tool_choice") or {}).get("type") in ("tool", "any")
        if result is None and tool and ("word_count" in prompt or forced):
            block = {"type": "tool_use", "id": "toolu_01", "name": tool[0], "input": {}}
            deltas = [{"type": "input_json_delta",
                       "partial_json": json.dumps({"text": self.quoted(prompt)})}]
            stop = "tool_use"
            message_content = [{"type": "tool_use", "id": "toolu_01", "name": tool[0],
                                "input": {"text": self.quoted(prompt)}}]
        elif "1 to 400" in prompt:
            numbers = " ".join(str(i) for i in range(1, 401))
            block = {"type": "text", "text": ""}
            deltas = [{"type": "text_delta", "text": numbers[i:i + 40]} for i in range(0, len(numbers), 40)]
            stop = "end_turn"
            message_content = [{"type": "text", "text": numbers}]
        else:
            text = self.answer_from(result)
            block = {"type": "text", "text": ""}
            deltas = [{"type": "text_delta", "text": text}]
            stop = "end_turn"
            message_content = [{"type": "text", "text": text}]
        usage = {"input_tokens": 10, "output_tokens": 5}
        if not req.get("stream"):
            await self.send_fixed(writer, 200, json.dumps({
                "id": "msg_01", "type": "message", "role": "assistant", "model": model,
                "content": message_content, "stop_reason": stop, "stop_sequence": None, "usage": usage}).encode())
            return
        events = [("message_start", {"type": "message_start", "message": {
                      "id": "msg_01", "type": "message", "role": "assistant", "model": model, "content": [],
                      "stop_reason": None, "stop_sequence": None, "usage": {"input_tokens": 10, "output_tokens": 1}}}),
                  ("content_block_start", {"type": "content_block_start", "index": 0, "content_block": block})]
        events += [("content_block_delta", {"type": "content_block_delta", "index": 0, "delta": d}) for d in deltas]
        events += [("content_block_stop", {"type": "content_block_stop", "index": 0}),
                   ("message_delta", {"type": "message_delta", "delta": {"stop_reason": stop, "stop_sequence": None},
                                      "usage": {"output_tokens": 5}}),
                   ("message_stop", {"type": "message_stop"})]
        pieces = [f"event: {e}\ndata: {json.dumps(d)}\n\n".encode() for e, d in events]
        await self.send_chunked(writer, pieces, "text/event-stream", 0.05 if len(deltas) > 10 else 0.02)

    async def responses(self, req, writer):
        result, prompt = None, ""
        items = req.get("input")
        items = [{"type": "message", "role": "user", "content": items}] if isinstance(items, str) else items or []
        for it in items:
            if it.get("type") == "function_call_output":
                out = it.get("output")
                result = out if isinstance(out, str) else json.dumps(out)
            elif it.get("type") == "message" and it.get("role") == "user":
                c = it.get("content")
                prompt = c if isinstance(c, str) else "".join(x.get("text", "") for x in c or [])
        tool = self.find_tool(req.get("tools"))
        rid = "resp_%06d" % len(self.requests)
        events = [{"type": "response.created", "response": {"id": rid}}]
        if result is None and tool and ("word_count" in prompt or req.get("tool_choice") == "required"):
            item = {"type": "function_call", "call_id": "call_01", "name": tool[0],
                    "arguments": json.dumps({"text": self.quoted(prompt)})}
            if tool[1] and tool[1] != tool[0]:
                item["namespace"] = tool[1]
            events.append({"type": "response.output_item.done", "item": item})
        else:
            text = self.answer_from(result)
            msg = {"type": "message", "role": "assistant", "id": "msg_01",
                   "content": [{"type": "output_text", "text": text}]}
            events += [{"type": "response.output_item.added", "item": dict(msg, content=[])},
                       {"type": "response.output_text.delta", "delta": text},
                       {"type": "response.output_item.done", "item": msg}]
        events.append({"type": "response.completed", "response": {"id": rid, "usage": {
            "input_tokens": 10, "input_tokens_details": None, "output_tokens": 5, "output_tokens_details": None,
            "total_tokens": 15}}})
        pieces = [f"event: {e['type']}\ndata: {json.dumps(e)}\n\n".encode() for e in events]
        await self.send_chunked(writer, pieces, "text/event-stream", 0.02)


# ---------------------------------------------------------------- gateway process
class GatewayProc:
    def __init__(self, upstream, workdir, name, keylog=None, cafile=None, extra=()):
        self.port = free_port()
        self.log = os.path.join(workdir, f"{name}_calls.jsonl")
        args = [sys.executable, GATEWAY, "--listen", f"127.0.0.1:{self.port}", "--upstream", upstream,
                "--log", self.log, *extra]
        if keylog:
            args += ["--keylog", keylog]
        if cafile:
            args += ["--cafile", cafile]
        self.proc = subprocess.Popen(args, stderr=subprocess.PIPE, text=True)
        line = self.proc.stderr.readline()
        if "listening" not in line:
            raise RuntimeError(f"gateway did not start: {line}")

    def records(self, event="call"):
        with open(self.log) as f:
            return [r for r in map(json.loads, f) if r.get("event") == event]

    def stop(self):
        self.proc.terminate()
        self.proc.wait(10)


# ---------------------------------------------------------------- raw client helpers
class RawClient:
    """HTTP/1.1 client on one kept-alive socket, so tests control framing and see arrival times."""

    def __init__(self, port):
        self.sock = socket.create_connection(("127.0.0.1", port))
        self.buf = b""

    def send(self, method, path, body=b"", headers=(), chunked=False, chunk=8191):
        h = [f"{method} {path} HTTP/1.1", "Host: 127.0.0.1", *headers]
        if chunked:
            h.append("Transfer-Encoding: chunked")
            self.sock.sendall(("\r\n".join(h) + "\r\n\r\n").encode())
            for i in range(0, len(body), chunk):
                piece = body[i:i + chunk]
                self.sock.sendall(f"{len(piece):x}\r\n".encode() + piece + b"\r\n")
            self.sock.sendall(b"0\r\n\r\n")
        else:
            if method not in ("GET", "HEAD") or body:
                h.append(f"Content-Length: {len(body)}")
            self.sock.sendall(("\r\n".join(h) + "\r\n\r\n").encode() + body)

    def _fill(self):
        data = self.sock.recv(65536)
        if not data:
            raise ConnectionError("closed")
        self.buf += data
        return time.time()

    def _until(self, sep):
        while sep not in self.buf:
            self._fill()
        i = self.buf.index(sep) + len(sep)
        out, self.buf = self.buf[:i], self.buf[i:]
        return out

    def _exact(self, n):
        while len(self.buf) < n:
            self._fill()
        out, self.buf = self.buf[:n], self.buf[n:]
        return out

    def response(self, method="POST", on_piece=None):
        while True:
            head = self._until(b"\r\n\r\n").decode("latin-1")
            status = int(head.split(" ", 2)[1])
            if status != 100:
                break
        hdr = {k.strip().lower(): v.strip() for k, _, v in
               (ln.partition(":") for ln in head.split("\r\n")[1:] if ln)}
        body = b""
        if method == "HEAD" or status in (204, 304):
            return status, hdr, body
        if "chunked" in hdr.get("transfer-encoding", ""):
            while True:
                size = int(self._until(b"\r\n").split(b";")[0], 16)
                if size == 0:
                    self._until(b"\r\n")
                    break
                piece = self._exact(size)
                if on_piece:
                    on_piece(piece, time.time())
                body += piece
                self._exact(2)
        else:
            body = self._exact(int(hdr.get("content-length", 0)))
        return status, hdr, body

    def close(self):
        self.sock.close()


# ---------------------------------------------------------------- checks
def check(results, name, passed, detail):
    results.append(dict(check=name, where="offline", passed=bool(passed), detail=detail))
    print(f"{'PASS' if passed else 'FAIL'}  {name}: {detail}", flush=True)


def run_core(results, workdir):
    fake = FakeEndpoint()
    gw = GatewayProc(f"http://127.0.0.1:{fake.port}/base", workdir, "core")
    rng = random.Random(7)
    def check_o1():
        """O1 request sizes"""
        c = RawClient(gw.port)
        bad = []
        cases = [(n, False) for n in SIZES] + [(n, True) for n in CHUNKED_SIZES]
        for n, chunked in cases:
            body = rng.randbytes(n)
            c.send("POST", "/size", body, ["Content-Type: application/octet-stream"], chunked=chunked,
                   chunk={0: 8191, 1: 1, 70_000: 1}.get(n, 777))
            status, _, resp = c.response()
            echo = json.loads(resp)
            if status != 200 or echo["len"] != n or echo["sha256"] != hashlib.sha256(body).hexdigest():
                bad.append((n, chunked, "upstream got", echo["len"]))
        c.close()
        time.sleep(0.2)
        logged = [r for r in gw.records() if r["path"] == "/size"]
        for (n, chunked), r in zip(cases, logged):
            if r["req_body_bytes"] != n or r["req_framing"] != ("chunked" if chunked else ("length" if True else "")):
                bad.append((n, chunked, "logged", r["req_body_bytes"], r["req_framing"]))
        prefix_ok = all(q["target"] == "/base/size" for q in fake.requests if q["target"].endswith("/size"))
        host_ok = all(q["headers"].get("host") == f"127.0.0.1:{fake.port}" for q in fake.requests)
        check(results, "O1 request sizes", not bad and len(logged) == len(cases) and prefix_ok and host_ok,
              f"{len(cases)} uploads from 0 B to 1 MB ({len(SIZES)} fixed-length, {len(CHUNKED_SIZES)} chunked): "
              f"logged length and upstream bytes exact for {len(cases) - len(bad)}/{len(cases)}; "
              f"path prefix kept {prefix_ok}; Host rewritten {host_ok}" + (f"; mismatches {bad[:3]}" if bad else ""))


    def check_o2():
        """O2 response sizes"""
        c = RawClient(gw.port)
        bad, cases = [], [(n, ch) for n in (0, 1, 1000, 1_000_000) for ch in (False, True)]
        for n, ch in cases:
            c.send("POST", f"/bigresp?n={n}&chunked={int(ch)}", b"{}")
            status, _, body = c.response()
            if body != random.Random(n).randbytes(n):
                bad.append((n, ch, "client got", len(body)))
        c.close()
        time.sleep(0.2)
        logged = [r for r in gw.records() if r["path"].startswith("/bigresp")]
        for (n, ch), r in zip(cases, logged):
            if r["resp_body_bytes"] != n or r["resp_framing"] != ("chunked" if ch else "length"):
                bad.append((n, ch, "logged", r["resp_body_bytes"]))
        check(results, "O2 response sizes", not bad and len(logged) == len(cases),
              f"{len(cases)} responses (0 B to 1 MB, fixed-length and chunked) relayed byte for byte and logged "
              f"exactly: {len(cases) - len(bad)}/{len(cases)}" + (f"; mismatches {bad[:3]}" if bad else ""))


    def check_o3():
        """O3 streaming"""
        c = RawClient(gw.port)
        arrivals = []
        c.send("POST", "/sse?events=8&gap=0.25&id=s1", b"{}")
        c.response(on_piece=lambda p, t: arrivals.append(t))
        c.close()
        sent = fake.sse_sent["s1"]
        lags = [a - s for a, s in zip(arrivals, sent)]
        r = [x for x in gw.records() if x["path"].startswith("/sse")][-1]
        first_before_last_sent = arrivals[0] < sent[-1] - 1.0
        ok = len(arrivals) == len(sent) == 8 and max(lags) < 0.1 and first_before_last_sent
        check(results, "O3 streaming", ok,
              f"8 events sent 250 ms apart over {sent[-1] - sent[0]:.2f} s; each reached the client "
              f"{min(lags) * 1e3:.1f} to {max(lags) * 1e3:.1f} ms after the endpoint sent it; first event arrived "
              f"{sent[-1] - arrivals[0]:.2f} s before the last was sent; gateway relayed {r['resp_pieces']} pieces")


    def check_o4():
        """O4 connection reuse"""
        before = len(fake.conns)
        c = RawClient(gw.port)
        for i in range(5):
            c.send("POST", "/size", b"x" * (i * 100))
            c.response()
        c.close()
        time.sleep(0.2)
        one = len(fake.conns) - before
        before = len(fake.conns)

        def worker():
            k = RawClient(gw.port)
            for i in range(3):
                k.send("POST", "/size", b"y" * 10)
                k.response()
            k.close()
        threads = [threading.Thread(target=worker) for _ in range(3)]
        [t.start() for t in threads]
        [t.join() for t in threads]
        time.sleep(0.2)
        three = len(fake.conns) - before
        calls = gw.records()
        by_conn = {}
        for r in calls:
            by_conn.setdefault(r["conn"], set()).add(r["up_conn"])
        one_up_each = all(len(v) == 1 for v in by_conn.values() if v != {None})
        check(results, "O4 connection reuse", one == 1 and three == 3 and one_up_each,
              f"5 sequential calls on one client connection used {one} upstream connection; 3 concurrent clients "
              f"with 3 calls each used {three}; every client connection kept a single upstream connection "
              f"{one_up_each}")


    def check_o5():
        """O5 reconnect"""
        before = len(fake.conns)
        c = RawClient(gw.port)
        c.send("POST", "/size", b"a")
        c.response()
        c.send("POST", "/close", b"{}")
        s1, h1, _ = c.response()
        survived = True
        try:
            c.send("POST", "/size", b"bb")
            s2, _, b2 = c.response()
        except ConnectionError:
            survived, s2, b2 = False, None, b"{}"
        c.close()
        time.sleep(0.2)
        recs = gw.records()[-3:]
        closes = [r for r in gw.records("up_close") if r["reason"] == "upstream asked to close"]
        ok = (s1 == 200 and h1.get("connection") == "close" and closes and survived and s2 == 200
              and json.loads(b2).get("len") == 2
              and len(fake.conns) - before == 2 and [r["up_new"] for r in recs] == [True, False, True])
        check(results, "O5 reconnect", ok,
              f"the endpoint answered with Connection: close and the gateway passed that header to the client "
              f"({h1.get('connection')}); upstream close logged {bool(closes)}; the client's next call on its kept "
              f"connection returned {s2}; upstream connections {len(fake.conns) - before}; up_new on the calls "
              f"{[r['up_new'] for r in recs]}")

    def check_o6():
        """O6 idle close"""
        before = len(fake.conns)
        c = RawClient(gw.port)
        c.send("POST", "/idleclose", b"{}")
        s1, _, _ = c.response()
        time.sleep(0.6)
        c.send("POST", "/size", b"ccc")
        s2, _, b2 = c.response()
        c.close()
        time.sleep(0.2)
        recs = gw.records()[-2:]
        idle = [r for r in gw.records("up_close") if r["reason"] == "upstream closed while idle"]
        ok = (s1 == 200 and s2 == 200 and json.loads(b2)["len"] == 3 and idle and len(fake.conns) - before == 2
              and [r["up_new"] for r in recs] == [True, True] and not [r for r in recs if r["error"]])
        check(results, "O6 idle close", ok,
              f"the endpoint dropped the kept connection 0.2 s after a call; the client's next call 0.6 s later on "
              f"the same client connection returned {s2}; idle close logged {bool(idle)}; upstream connections "
              f"{len(fake.conns) - before}; up_new on the two calls {[r['up_new'] for r in recs]}")


    def check_o7():
        """O7 client close"""
        c = RawClient(gw.port)
        c.send("POST", "/size", b"z", ["Connection: close"])
        c.response()
        time.sleep(0.3)
        c.close()
        last_up = gw.records("up_close")[-1]
        check(results, "O7 client close", last_up["reason"] == "agent asked to close" and fake.conns[-1]["eof"],
              f"client sent Connection: close; gateway closed its upstream connection ({last_up['reason']}); "
              f"endpoint saw the close {fake.conns[-1]['eof']}")


    def check_o8():
        """O8 no-body responses"""
        before = len(fake.conns)
        c = RawClient(gw.port)
        c.send("HEAD", "/head")
        s_head, h_head, b_head = c.response("HEAD")
        c.send("POST", "/nocontent", b"{}")
        s_204, _, _ = c.response()
        c.send("POST", "/size", b"after")
        s_after, _, b_after = c.response()
        c.close()
        time.sleep(0.2)
        ok = (s_head == 200 and b_head == b"" and h_head.get("content-length") == "12345" and s_204 == 204
              and s_after == 200 and json.loads(b_after)["len"] == 5 and len(fake.conns) - before == 1)
        check(results, "O8 no-body responses", ok,
              f"HEAD {s_head} with Content-Length {h_head.get('content-length')} and no body, then 204, then a sized "
              f"call on the same connection: {s_after}, one upstream connection {len(fake.conns) - before == 1}")


    def check_o9():
        """O9 expect-continue"""
        c = RawClient(gw.port)
        body = rng.randbytes(50_000)
        head = (f"POST /size HTTP/1.1\r\nHost: x\r\nContent-Length: {len(body)}\r\nExpect: 100-continue\r\n\r\n")
        c.sock.sendall(head.encode())
        interim = c._until(b"\r\n\r\n")
        c.sock.sendall(body)
        status, _, resp = c.response()
        c.close()
        time.sleep(0.2)
        r = gw.records()[-1]
        ok = interim.startswith(b"HTTP/1.1 100") and status == 200 and json.loads(resp)["len"] == 50_000 \
            and r["req_body_bytes"] == 50_000 and r["expect_continue"]
        check(results, "O9 expect-continue", ok,
              f"client got 100 Continue from the gateway, sent 50000 bytes, endpoint received "
              f"{json.loads(resp)['len']}, logged {r['req_body_bytes']}")

    def check_o12():
        """O12 endpoint drop"""
        c = RawClient(gw.port)
        received = []
        c.send("POST", "/sse?events=4&gap=0.05&cut=2&id=drop", b"{}")
        try:
            c.response(on_piece=lambda p, t: received.append(p))
            closed = False
        except ConnectionError:
            closed = True
        c.close()
        time.sleep(0.3)
        r = [x for x in gw.records() if x["path"].startswith("/sse?") and "id=drop" in x["path"]][-1]
        sent = sum(len(p) for p in received)
        ok = (closed and r["error_side"] == "upstream" and r["resp_body_bytes"] == sent == 2 * 28
              and r["resp_pieces"] == 2)
        check(results, "O12 endpoint drop", ok,
              f"the endpoint sent 2 of 4 events and dropped the connection; the client got {sent} bytes and then a "
              f"closed connection {closed}; the gateway logged {r['resp_body_bytes']} bytes in {r['resp_pieces']} "
              f"pieces, error side {r['error_side']} ({r['error']})")

    def check_o13():
        """O13 agent close"""
        c = RawClient(gw.port)
        c.send("POST", "/sse?events=3&gap=0.05&endgap=0.3&id=early", b"{}")
        got = b""
        while got.count(b"event: tick") < 3:
            got += c.sock.recv(65536)
        c.sock.close()
        time.sleep(0.8)
        r = [x for x in gw.records() if x["path"].startswith("/sse?") and "id=early" in x["path"]][-1]
        up = [x for x in gw.records("up_close") if x["up_conn"] == r["up_conn"]]
        ok = r["error_side"] == "agent" and r["resp_body_bytes"] == 3 * 28 and up \
            and up[-1]["reason"] == "agent connection failed during the call"
        check(results, "O13 agent close", ok,
              f"the client closed after the 3rd event, before the end of the stream; the gateway logged "
              f"{r['resp_body_bytes']} bytes, error side {r['error_side']} ({r['error']}); upstream connection "
              f"closed: {up[-1]['reason'] if up else None}")

    try:
        for fn in (check_o1, check_o2, check_o3, check_o4, check_o5, check_o6, check_o7, check_o8, check_o9,
                   check_o12, check_o13):
            try:
                fn()
            except Exception as e:
                check(results, fn.__doc__, False, f"stopped with {type(e).__name__}: {e}")
    finally:
        gw.stop()


def run_tls(results, workdir):
    if not shutil.which("openssl"):
        results.append(dict(check="O10 TLS key log", where="offline", passed=None,
                            detail="skipped: openssl command not found"))
        print("SKIP  O10 TLS key log: openssl command not found")
        return
    key, crt = os.path.join(workdir, "fake.key"), os.path.join(workdir, "fake.crt")
    subprocess.run(["openssl", "req", "-x509", "-newkey", "rsa:2048", "-nodes", "-keyout", key, "-out", crt,
                    "-days", "1", "-subj", "/CN=localhost", "-addext", "subjectAltName=DNS:localhost"],
                   check=True, capture_output=True)
    ctx = ssl.create_default_context(ssl.Purpose.CLIENT_AUTH)
    ctx.load_cert_chain(crt, key)
    ctx.set_alpn_protocols(["http/1.1"])
    fake = FakeEndpoint(ctx)
    keylog = os.path.join(workdir, "tls.keys")
    gw = GatewayProc(f"https://localhost:{fake.port}", workdir, "tls", keylog=keylog, cafile=crt)
    try:
        c = RawClient(gw.port)
        sizes = [0, 12345, 1_000_000]
        echoed = []
        for n in sizes:
            c.send("POST", "/size", b"k" * n)
            echoed.append(json.loads(c.response()[2])["len"])
        c.close()
        time.sleep(0.2)
        logged = [r["req_body_bytes"] for r in gw.records()]
        with open(keylog) as f:
            labels = {ln.split()[0] for ln in f if ln.strip() and not ln.startswith("#")}
        ok = echoed == sizes and logged == sizes and len(fake.conns) == 1 and \
            ("CLIENT_TRAFFIC_SECRET_0" in labels or "CLIENT_RANDOM" in labels)
        check(results, "O10 TLS key log", ok,
              f"HTTPS upstream with verified certificate: sizes {sizes} logged {logged}; one TLS connection "
              f"{len(fake.conns) == 1}; key log lines {sorted(labels)}")
    finally:
        gw.stop()


def run_hold(results, workdir):
    fake = FakeEndpoint()
    flag = os.path.join(workdir, "hold.flag")
    gw = GatewayProc(f"http://127.0.0.1:{fake.port}", workdir, "hold", extra=["--hold-until", flag, "--hold-max", "20"])
    gw2 = GatewayProc(f"http://127.0.0.1:{fake.port}", workdir, "hold_timeout",
                      extra=["--hold-until", flag + ".never", "--hold-max", "1"])
    try:
        timer = threading.Timer(1.5, lambda: open(flag, "w").close())
        t0 = time.time()
        timer.start()
        c = RawClient(gw.port)
        c.send("POST", "/size", b"h" * 5000)
        first = json.loads(c.response()[2])["len"]
        t_first = time.time() - t0
        t1 = time.time()
        c.send("POST", "/size", b"i" * 7)
        second = json.loads(c.response()[2])["len"]
        t_second = time.time() - t1
        c.close()
        c2 = RawClient(gw2.port)
        t2 = time.time()
        c2.send("POST", "/size", b"j" * 11)
        third = json.loads(c2.response()[2])["len"]
        t_third = time.time() - t2
        c2.close()
        time.sleep(0.2)
        held = gw.records("hold")
        held2 = gw2.records("hold")
        sizes = [r["req_body_bytes"] for r in gw.records()] + [r["req_body_bytes"] for r in gw2.records()]
        ok = (first == 5000 and second == 7 and third == 11 and sizes == [5000, 7, 11] and 1.4 <= t_first < 5
              and t_second < 0.5 and 0.9 <= t_third < 3 and len(held) == 1 and held[0]["released"] == "file"
              and len(held2) == 1 and held2[0]["released"] == "timeout")
        check(results, "O14 warm-up hold", ok,
              f"first request answered after {t_first:.2f} s (file created at 1.5 s), hold released by "
              f"{held[0]['released'] if held else None}; second request {t_second * 1e3:.0f} ms, not held; with no "
              f"file and --hold-max 1 the request was answered after {t_third:.2f} s, released by "
              f"{held2[0]['released'] if held2 else None}; logged sizes {sizes}")
    finally:
        gw.stop()
        gw2.stop()


def run_mcp(results, workdir):
    log = os.path.join(workdir, "tools.jsonl")
    ws = os.path.join(workdir, "ws")
    env = dict(os.environ, UT_TOOL_LOG=log, UT_WORKSPACE=ws)
    p = subprocess.Popen([sys.executable, MCP], stdin=subprocess.PIPE, stdout=subprocess.PIPE, env=env)
    args = {"text": "alpha beta gamma delta été"}
    call = {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "word_count", "arguments": args}}
    msgs = [{"jsonrpc": "2.0", "id": 1, "method": "initialize",
             "params": {"protocolVersion": "2025-06-18", "capabilities": {},
                        "clientInfo": {"name": "t", "version": "1"}}},
            {"jsonrpc": "2.0", "method": "notifications/initialized"},
            {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}, call,
            {"jsonrpc": "2.0", "id": 4, "method": "tools/call",
             "params": {"name": "read_file", "arguments": {"path": "../../etc/passwd"}}}]
    lines = [json.dumps(m).encode() for m in msgs]
    out, _ = p.communicate(b"\n".join(lines) + b"\n", timeout=20)
    replies = [json.loads(x) for x in out.splitlines()]
    with open(log) as f:
        recs = [json.loads(x) for x in f]
    tc = [r for r in recs if r["event"] == "tool_call"]
    want_args = len(json.dumps(args, separators=(",", ":"), ensure_ascii=False).encode())
    ok = (len(replies) == 4 and replies[2]["result"]["content"][0]["text"] == "5"
          and len(replies[1]["result"]["tools"]) == 4 and len(tc) == 2
          and tc[0]["args_bytes"] == want_args and tc[0]["msg_bytes"] == len(lines[3])
          and tc[0]["result_bytes"] == len(out.splitlines()[2]) and tc[1]["is_error"] is True)
    check(results, "O11 MCP tool log", ok,
          f"initialize, tools/list and two tools/call answered; word_count returned "
          f"{replies[2]['result']['content'][0]['text']}; args_bytes {tc[0]['args_bytes'] if tc else None} "
          f"(expected {want_args}), msg_bytes exact {bool(tc) and tc[0]['msg_bytes'] == len(lines[3])}; "
          f"path outside the workspace refused {bool(tc) and tc[-1]['is_error']}")


# ---------------------------------------------------------------- real agent clients against the fake endpoint
def run_agent(results, workdir, which):
    exe = shutil.which("claude" if which == "claude" else "codex")
    name = "A1 Claude Code via gateway" if which == "claude" else "A2 Codex via gateway"
    if not exe:
        results.append(dict(check=name, where="offline", passed=None, detail="skipped: command not found"))
        print(f"SKIP  {name}: command not found")
        return
    fake = FakeEndpoint()
    run = os.path.join(workdir, which)
    home, ws = os.path.join(run, "home"), os.path.join(run, "ws")
    os.makedirs(home, exist_ok=True)
    os.makedirs(ws, exist_ok=True)
    tool_log = os.path.join(run, "tools.jsonl")
    upstream = f"http://127.0.0.1:{fake.port}" + ("/anthropic" if which == "claude" else "")
    gw = GatewayProc(upstream, run, which)
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(("ANTHROPIC_", "OPENAI_", "CODEX_", "CLAUDE_"))
           and k.lower() not in ("http_proxy", "https_proxy", "all_proxy")}
    env.update(HOME=home, UT_TOOL_LOG=tool_log, UT_WORKSPACE=ws, NO_PROXY="127.0.0.1,localhost",
               no_proxy="127.0.0.1,localhost")
    mcp = {"command": sys.executable, "args": [MCP], "env": {"UT_TOOL_LOG": tool_log, "UT_WORKSPACE": ws}}
    if which == "claude":
        cfg = os.path.join(run, "mcp.json")
        with open(cfg, "w") as f:
            json.dump({"mcpServers": {"bench": dict(mcp, type="stdio")}}, f)
        env.update(ANTHROPIC_BASE_URL=f"http://127.0.0.1:{gw.port}", ANTHROPIC_AUTH_TOKEN="offline-test-token",
                   ANTHROPIC_MODEL="deepseek-flash", ANTHROPIC_DEFAULT_OPUS_MODEL="deepseek-flash",
                   ANTHROPIC_DEFAULT_SONNET_MODEL="deepseek-flash", ANTHROPIC_DEFAULT_HAIKU_MODEL="deepseek-flash",
                   CLAUDE_CODE_SUBAGENT_MODEL="deepseek-flash", CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC="1")
        cmd = [exe, "-p", AGENT_PROMPT, "--mcp-config", cfg, "--strict-mcp-config",
               "--allowedTools", "mcp__bench__word_count", "--output-format", "json", "--max-turns", "4"]
    else:
        codex_home = os.path.join(home, ".codex")
        os.makedirs(codex_home, exist_ok=True)
        with open(os.path.join(codex_home, "config.toml"), "w") as f:
            f.write(codex_config(f"http://127.0.0.1:{gw.port}", mcp))
        env.update(CODEX_HOME=codex_home, DEEPSEEK_API_KEY="offline-test-token")
        cmd = [exe, "exec", "--skip-git-repo-check", "--ephemeral", "-C", ws,
               "-o", os.path.join(run, "last_message.txt"), AGENT_PROMPT]
    t0 = time.time()
    try:
        p = subprocess.run(cmd, env=env, cwd=ws, stdin=subprocess.DEVNULL, capture_output=True, text=True,
                           timeout=240)
        code, out, err = p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired as e:
        code, out, err = "timeout", e.stdout or "", e.stderr or ""
    took = time.time() - t0
    time.sleep(0.3)
    gw.stop()
    calls = gw.records()
    with open(os.path.join(run, "agent_stdout.txt"), "w") as f:
        f.write(str(out))
    with open(os.path.join(run, "agent_stderr.txt"), "w") as f:
        f.write(str(err))
    tools = []
    if os.path.exists(tool_log):
        with open(tool_log) as f:
            tools = [r for r in map(json.loads, f) if r["event"] == "tool_call"]
    answer = ""
    if which == "claude":
        try:
            answer = str(json.loads(out).get("result", ""))
        except ValueError:
            answer = ""
    elif os.path.exists(os.path.join(run, "last_message.txt")):
        with open(os.path.join(run, "last_message.txt")) as f:
            answer = f.read()
    model_calls = [r for r in calls if r["path"].split("?")[0].endswith(("/v1/messages", "/responses"))]
    streamed = [r for r in model_calls if (r["resp_content_type"] or "").startswith("text/event-stream")]
    agent_conns = sorted({r["conn"] for r in calls})
    up_conns = sorted({r["up_conn"] for r in calls if r["up_conn"]})
    per_conn = {c: len([r for r in calls if r["conn"] == c]) for c in agent_conns}
    sizes_ok = all(r["req_body_bytes"] == q["length"] for r, q in zip(
        [r for r in calls if r["req_framing"] != "none"], [q for q in fake.requests if q["method"] != "GET"]))
    agent_closes = [r for r in calls if r["error"] and r.get("error_side") == "agent"]
    errors = [r for r in calls if r["error"] and r.get("error_side") != "agent"]
    ok = (code == 0 and len(model_calls) >= 2 and streamed and len(tools) >= 1 and tools[0]["tool"] == "word_count"
          and tools[0]["args_bytes"] > 0 and re.findall(r"\d+", answer) == ["4"] and len(up_conns) == len(agent_conns)
          and sizes_ok and not errors)
    paths = sorted({f"{r['method']} {r['path'].split('?')[0]}" for r in calls})
    check(results, name, ok,
          f"exit {code} in {took:.0f} s; {len(calls)} calls through the gateway ({', '.join(paths)}), "
          f"{len(streamed)} streamed; {len(agent_conns)} agent connections, {len(up_conns)} upstream connections, "
          f"calls per agent connection {list(per_conn.values())}; logged request sizes equal the bytes the endpoint "
          f"received {sizes_ok}; tool calls logged {[(t['tool'], t['args_bytes']) for t in tools]}; "
          f"answer {(answer or '').strip()[-60:]!r}; calls the agent closed itself {len(agent_closes)}; other "
          f"errors {[r['error'] for r in errors]}")


def codex_config(base_url, mcp):
    env = ", ".join(f'{k} = "{v}"' for k, v in mcp["env"].items())
    return (f'model = "deepseek-flash"\n'
            f'model_provider = "deepseek"\n'
            f'approval_policy = "never"\n'
            f'sandbox_mode = "workspace-write"\n'
            f'web_search = "disabled"\n\n'
            f'[model_providers.deepseek]\n'
            f'name = "deepseek"\n'
            f'base_url = "{base_url}"\n'
            f'env_key = "DEEPSEEK_API_KEY"\n'
            f'wire_api = "responses"\n\n'
            f'[mcp_servers.bench]\n'
            f'command = "{mcp["command"]}"\n'
            f'args = {json.dumps(mcp["args"])}\n'
            f'env = {{ {env} }}\n'
            f'default_tools_approval_mode = "approve"\n')


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--agents", action="store_true", help="also run the installed agent clients")
    ap.add_argument("--out", default=None, help="folder for logs and the JSON report")
    args = ap.parse_args(argv)
    workdir = args.out or tempfile.mkdtemp(prefix="ut_offline_")
    os.makedirs(workdir, exist_ok=True)
    results = []
    run_core(results, workdir)
    run_tls(results, workdir)
    run_hold(results, workdir)
    run_mcp(results, workdir)
    if args.agents:
        run_agent(results, workdir, "claude")
        run_agent(results, workdir, "codex")
    with open(os.path.join(workdir, "offline_check.json"), "w") as f:
        json.dump(results, f, indent=1)
    failed = [r["check"] for r in results if r["passed"] is False]
    print(f"\nOFFLINE CHECK: {'PASS' if not failed else 'FAIL'}"
          + (f" (failed: {', '.join(failed)})" if failed else "") + f"; report {workdir}/offline_check.json")
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile harness/README.md
# Collection harness (benign runs)

Testbed that runs benign agent tasks and records, for every clean run, the exact byte length of each model call and
each tool call, next to a packet capture of that run. It runs on a Linux VM with root, because packet capture and
per-run network isolation need control of the operating system. Everything after collection runs in Colab.

| File | Role |
|---|---|
| `call_gateway.py` | Gateway the agent calls instead of the model endpoint. Logs the exact request and response body length of every call (JSON lines), forwards the request unchanged apart from Host and the upstream path prefix, streams the response back as it arrives, keeps one upstream connection per agent connection. Upstream TLS keys go to a key log for ground-truth labelling only. `--hold-until` is used only to prepare a warm home. |
| `homes.py` | Warm template homes, one per agent version, and their private copy into each run's home. |
| `mcp_tools.py` | Local MCP tool server (stdio) with benign workspace tools. Logs each tool call's argument size at the tool boundary. |
| `netns.py` | One network namespace per run, with NAT, public DNS, containment rules and a capture on the run's own interface. The veth pair takes the uplink's MTU and has segmentation and receive offloads off, and tcpdump writes every packet as it arrives, so the capture holds wire-size packets and no packet is lost when it stops. |
| `offline_check.py` | Checks that need no network or root: exact sizes from 0 B to 1 MB, unbuffered streaming, connection reuse, reconnects, TLS key log, MCP tool log, and (with `--agents`) Claude Code and Codex against a local fake endpoint. |
| `vm_gate.py` | Checks on the VM before any collection: isolation and capture, DeepSeek Anthropic and Responses APIs through the gateway, key-log decryption, both agents through the gateway, and the traffic of each agent straight to DeepSeek compared with its traffic through the gateway. |
| `vm_setup.sh` | Installs the system packages (including ethtool), Node.js 22 and the pinned agent versions on Ubuntu 24.04. |

## VM gate

    git clone https://github.com/anasbiswas1/unexplained-traffic.git
    cd unexplained-traffic
    sudo bash harness/vm_setup.sh
    export DEEPSEEK_API_KEY=...        # kept in the environment only, never written to disk
    sudo -E python3 harness/vm_gate.py

Output goes to `data/harness/vm_gate/<time>/` (git-ignored): captures, call logs, tool logs, agent output, a
`net.json` per run (uplink, MTU, offload states, capture counts) and `vm_gate.json`. Key logs are deleted when the gate
ends.

## Warm homes

Benign runs represent a returning user with a warm home, not a first launch. Each agent version has one template home,
made at the start of each gate or collection session by one trivial task run from an empty home and saved under
`data/harness/homes/<agent>-<version>/` with a `manifest.json`. Every run gets its own copy of the template before the
agent starts; templates are never shared between runs. All runs therefore carry the template's installation identity,
as one user's sessions would.

- Codex syncs its plugin catalogue (`github.com/openai/plugins`) in the background at every launch and keeps the result
  only if the sync finishes before the agent exits. From an empty home a short run downloads about 25 MB and throws it
  away, so every such run downloads it again. The warm-up therefore holds its first model call at the gateway until
  the sync has finished (`.codex/.tmp/plugins.sha` exists), and the template is saved only then. A run from that home
  checks the remote commit (`git ls-remote`, about 40 KB) and fetches again only if the catalogue has changed upstream
  since the template was made.
- Claude Code showed no comparable first-launch download in testing (about 14 KB, as on a warm launch); its template
  is made the same way, without the hold.
- No template file may contain the API key.

V6, V7 and V8 report bytes up and down per remote host for every run and fail a run that downloads more than 1,000,000
bytes from any host other than the model host.

## Agent settings on DeepSeek

- Claude Code, from DeepSeek's Claude Code guide: `ANTHROPIC_BASE_URL` (the gateway, which forwards to
  `https://api.deepseek.com/anthropic`), `ANTHROPIC_AUTH_TOKEN`, `ANTHROPIC_MODEL=deepseek-flash[1m]`,
  `ANTHROPIC_DEFAULT_OPUS_MODEL=deepseek-flash[1m]`, `ANTHROPIC_DEFAULT_SONNET_MODEL=deepseek-flash[1m]`,
  `ANTHROPIC_DEFAULT_HAIKU_MODEL=deepseek-flash`. Local MCP tools reach the model as ordinary tool definitions.
- Codex, from DeepSeek's Codex guide: provider `base_url` (the gateway, which forwards to `https://api.deepseek.com`),
  `wire_api = "responses"`, model `deepseek-flash`; the key is read from `DEEPSEEK_API_KEY` through `env_key`.

## Fixed rules

- Benign tasks only. No real credentials and no real user data in any run.
- The decrypted view of a capture exists only to label which bytes were real content, as a check on the byte-budget
  analysis. It is never an input to the detector.
- Key logs are deleted with the testbed and never released. Captures, key logs and call logs live under `data/` and
  are never committed.

In [ ]:
import py_compile
for f_ in ('homes.py', 'call_gateway.py', 'vm_gate.py', 'offline_check.py'):
    py_compile.compile(f'harness/{f_}', doraise=True)
for f_ in ('homes.py', 'call_gateway.py', 'vm_gate.py', 'offline_check.py', 'README.md'):
    print(f'{f_:18s} {os.path.getsize(f"harness/{f_}"):>8,d} bytes')

## Stage B. Unit checks and the offline gateway check

In [ ]:
import importlib
sys.path.insert(0, 'harness')
import netns, vm_gate, homes
importlib.reload(netns)
importlib.reload(vm_gate)
importlib.reload(homes)
C, E = '10.200.13.2', '1.2.3.4'
UNIT = []

# U1 to U5: upload measures and the upstream-drop rule (as in 02b)
seg = [['0', C, '1', '1000'], ['0', C, '1001', '1000'], ['0', C, '1001', '1000'], ['0', C, '2001', '500'],
       ['0', E, '1', '700'], ['1', C, '1', '1000'], ['1', C, '501', '1000'], ['2', C, '', '']]
rec = [['0', C, '1495,995'], ['1', C, '1495'], ['0', E, '695']]
m = vm_gate.upload_measures(seg, rec, C)
UNIT.append(('U1 upload with a full resend', (m['0']['up_raw'], m['0']['up_tcp'], m['0']['resent'], m['0']['up_tls'],
                                              m['0']['down_raw']) == (3500, 2500, 1000, 2500, 700), m['0']))
UNIT.append(('U2 upload with a partial resend', (m['1']['up_raw'], m['1']['up_tcp'], m['1']['resent'],
                                                  m['1']['up_tls']) == (2000, 1500, 500, 1500), m['1']))
rows = [['99.0', E, '443', '40000', 'True', 'False'],
        ['100.52', E, '443', '40000', 'True', 'False'], ['100.53', C, '40000', '443', '1', '0'],
        ['200.10', C, '40001', '443', 'True', 'False'], ['200.11', E, '443', '40001', 'False', 'True'],
        ['300.20', E, '443', '40002', '0', '1'], ['400.20', E, '443', '40003', '1', '0']]
ev = vm_gate.close_rows_to_events(rows, C)
base = dict(seq=1, status=200, path='/v1/messages?beta=true')
calls = [dict(base, conn=1, t=100.0, t_resp_head=100.5, resp_body_bytes=0, up_local=f'{C}:40000',
              error='IncompleteReadError: 0 bytes read', error_side='upstream'),
         dict(base, conn=2, t=200.0, t_resp_head=200.05, resp_body_bytes=0, up_local=f'{C}:40001',
              error='ConnectionResetError: x', error_side='upstream'),
         dict(base, conn=3, t=300.0, t_resp_head=300.1, resp_body_bytes=900, up_local=f'{C}:40002', error=None,
              error_side=None),
         dict(base, conn=4, t=300.0, t_resp_head=300.1, resp_body_bytes=900, up_local=f'{C}:40002',
              error='IncompleteReadError: y', error_side='upstream'),
         dict(base, conn=5, t=400.0, t_resp_head=400.1, resp_body_bytes=84, up_local=f'{C}:40003',
              error='ConnectionResetError: Connection lost', error_side='agent')]
d = vm_gate.upstream_drops(calls, ev)
UNIT.append(('U3 endpoint drop after the head, before any body',
             len(d) == 2 and d[0]['conn'] == 1 and d[0]['phase'] == 'after the 200 response head, before any body'
             and d[0]['closed_by'] == 'endpoint FIN' and d[0]['seconds_after_head'] == 0.02, d[:1]))
UNIT.append(('U4 endpoint drop inside the body', len(d) == 2 and d[1]['conn'] == 4
             and d[1]['phase'] == 'inside the body, after 900 bytes' and d[1]['closed_by'] == 'endpoint RST', d[1:]))
UNIT.append(('U5 client close, agent close and clean calls are not drops',
             {x['conn'] for x in d}.isdisjoint({2, 3, 5}), [x['conn'] for x in d]))

# U6: download resends come from segments tshark marks as resent, endpoint side only
m6 = vm_gate.upload_measures(seg, rec, C, flagged=[['0', E, '1408'], ['0', C, '1000'], ['1', E, '700']])
UNIT.append(('U6 download resends', (m6['0']['down_resent'], m6['1']['down_resent'], m6['0']['resent']) ==
             (1408, 700, 1000), {k: v['down_resent'] for k, v in m6.items()}))

# U7: uplink, route mtu attribute and link MTU parsing
gce = "default via 10.128.0.1 dev ens4 proto dhcp src 10.128.0.2 metric 100"
link = ("2: ens4: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1460 qdisc mq state UP mode DEFAULT group default qlen 1000"
        "\\    link/ether 42:01:0a:80:00:02 brd ff:ff:ff:ff:ff:ff")
r7 = (netns.parse_default_route(gce), netns.parse_default_route("default via 10.0.0.1 dev eth0 mtu 1400"),
      netns.parse_default_route("default via 10.0.0.1 dev eth0 mtu lock 1380 metric 5"), netns.parse_link_mtu(link))
UNIT.append(('U7 uplink and MTU parsing', r7 == (('ens4', None), ('eth0', 1400), ('eth0', 1380), 1460), r7))

# U8: offload states
ON = ("Features for utn13:\nrx-checksumming: on\ntx-checksumming: on\nscatter-gather: on\n"
      "tcp-segmentation-offload: on\n\ttx-tcp-segmentation: on\ngeneric-segmentation-offload: on\n"
      "generic-receive-offload: off\nlarge-receive-offload: off [fixed]\n")
OFF = ON.replace("tcp-segmentation-offload: on", "tcp-segmentation-offload: off").replace(
    "generic-segmentation-offload: on", "generic-segmentation-offload: off")
r8 = (netns.offloads_off(netns.parse_offloads(ON)), netns.offloads_off(netns.parse_offloads(OFF)),
      netns.offloads_off(netns.parse_offloads(OFF.replace("generic-receive-offload: off\n", ""))))
UNIT.append(('U8 offload states', r8 == (False, True, False), r8))

# U9: tcpdump closing counts
done = "tcpdump: listening on utn13\n1875 packets captured\n1875 packets received by filter\n0 packets dropped by kernel"
tail = "1616 packets captured\n1737 packets received by filter\n0 packets dropped by kernel"
drop = "900 packets captured\n950 packets received by filter\n50 packets dropped by kernel"
r9 = [netns.parse_capture_stats(t)['complete'] for t in (done, tail, drop, "")]
UNIT.append(('U9 capture counts', r9 == [True, False, False, False], r9))

# U10: wire verdict on a clean capture and on each fault
offs = {'uth13': netns.parse_offloads(OFF), 'utn13': netns.parse_offloads(OFF)}
clean = dict(mtu=1460, uplink='ens4', uplink_mtu=1460, host_if_mtu=1460, ns_if_mtu=1460, offloads=offs,
             icmp_frag_needed=0, icmp_next_hop_mtu=[], packets=900, largest_packet=1460, up_resent=0,
             down_resent=1408, capture=dict(captured=900, received=900, dropped_kernel=0, dropped_interface=None,
                                            complete=True))
faults = dict(mtu_mismatch=dict(host_if_mtu=1500, ns_if_mtu=1500),
              offloads_on=dict(offloads={'utn13': netns.parse_offloads(ON)}),
              icmp=dict(icmp_frag_needed=2, icmp_next_hop_mtu=['1460']),
              block=dict(largest_packet=65212),
              resent=dict(up_resent=12852),
              tail=dict(capture=dict(captured=1616, received=1737, dropped_kernel=0, dropped_interface=None,
                                     complete=False)))
v_clean = vm_gate.wire_verdict(clean)
v_edge = vm_gate.wire_verdict(dict(clean, up_resent=1420))
v_faults = {k: vm_gate.wire_verdict(dict(clean, **f)) for k, f in faults.items()}
UNIT.append(('U10 wire verdict', v_clean[0] and v_edge[0] and not any(v[0] for v in v_faults.values())
             and all(len(v[1]) == 1 for v in v_faults.values()),
             dict(clean=v_clean, at_tolerance=v_edge[0], faults={k: v[1] for k, v in v_faults.items()})))
# U11: bytes per host from capture rows (TLS name, bare address, UDP and DNS, ICMP-quoted values)
tcp_rows = [['0', C, E, '40000', '443', '0'], ['0', C, E, '40000', '443', '500'], ['0', E, C, '443', '40000', '2000'],
            ['1', C, '5.6.7.8', '40001', '443', '100'], ['1', '5.6.7.8', C, '443', '40001', '1500000'],
            ['2', C, '9.9.9.9', '40002', '8080', '30'], ['2', '9.9.9.9', C, '8080', '40002', '70']]
udp_rows = [[C, '1.1.1.1', '50000', '53', '40'], ['1.1.1.1', C, '53', '50000', '120'],
            [C, '7.7.7.7', '50001', '443', '1208']]
ht = vm_gate.host_table_from_rows(tcp_rows, udp_rows, C, {'0': 'api.deepseek.com', '1': 'github.com'})
want = {'api.deepseek.com': dict(conns=1, up=500, down=2000), 'github.com': dict(conns=1, up=100, down=1500000),
        '9.9.9.9:8080': dict(conns=1, up=30, down=70), 'dns 1.1.1.1': dict(conns=1, up=32, down=112),
        'udp 7.7.7.7:443': dict(conns=1, up=1200, down=0)}
UNIT.append(('U11 bytes per host', ht == want, ht))

# U12: background download rule: model host exempt, other hosts over 1,000,000 bytes fail
bg = vm_gate.background_downloads(dict(ht, **{'api.deepseek.com': dict(conns=1, up=1, down=5_000_000)}))
edge = vm_gate.background_downloads({'x.example': dict(conns=1, up=0, down=1_000_000)})
UNIT.append(('U12 background download rule', bg == [('github.com', 1500000)] and edge == [], dict(flagged=bg, at_limit=edge)))

# U13 to U16: template homes
import tempfile, shutil as _sh
tmp = tempfile.mkdtemp(prefix='u_homes_')
h = os.path.join(tmp, 'home')
os.makedirs(os.path.join(h, '.codex', '.tmp', 'plugins-clone-abc', '.git'))
s13a = homes.codex_sync_state(h)
_sh.rmtree(os.path.join(h, '.codex', '.tmp', 'plugins-clone-abc'))
os.makedirs(os.path.join(h, '.codex', '.tmp', 'plugins', '.git'))
open(os.path.join(h, '.codex', '.tmp', 'plugins.sha'), 'w').write('0722921d5542abcdef\n')
s13b = homes.codex_sync_state(h)
UNIT.append(('U13 Codex sync state', (not s13a['synced'] and s13a['leftover_clones'] == ['plugins-clone-abc']
                                     and s13b['synced'] and s13b['sha'] == '0722921d5542abcdef'), dict(unfinished=s13a, finished=s13b)))
open(os.path.join(h, '.codex', 'auth.json'), 'w').write('{"key": "sk-unit-test-secret-0001"}')
p14a = homes.template_problems('codex', h, 'sk-unit-test-secret-0001')
os.remove(os.path.join(h, '.codex', 'auth.json'))
p14b = homes.template_problems('codex', h, 'sk-unit-test-secret-0001')
UNIT.append(('U14 template checks', len(p14a) == 1 and 'API key found' in p14a[0] and p14b == [], dict(with_key=p14a, clean=p14b)))
root = os.path.join(tmp, 'templates')
dest = homes.save_template(h, root, 'codex', '0.163.0', dict(note='unit'), 'sk-unit-test-secret-0001')
t = homes.load_template(root, 'codex', '0.163.0')
run_home = os.path.join(tmp, 'run1', 'home')
homes.copy_template(t['home'], run_home)
open(os.path.join(run_home, '.codex', 'run_only.txt'), 'w').write('x')
private = not os.path.exists(os.path.join(t['home'], '.codex', 'run_only.txt'))
try:
    homes.copy_template(t['home'], run_home)
    refused = False
except RuntimeError:
    refused = True
UNIT.append(('U15 template save and private copy', (dest.endswith('codex-0.163.0') and t['codex_plugins']['synced']
             and private and refused and os.path.isfile(os.path.join(run_home, '.codex', '.tmp', 'plugins.sha'))),
             dict(template=dest, private_copy=private, second_copy_refused=refused)))
v16 = [homes.parse_version(x) for x in ('2.1.302 (Claude Code)', 'codex-cli 0.163.0')]
UNIT.append(('U16 agent version', v16 == ['2.1.302', '0.163.0'], v16))
_sh.rmtree(tmp)

for name, ok, value in UNIT:
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {value}")
UNIT_PASS = all(ok for _, ok, _ in UNIT)

In [ ]:
import json, subprocess
import pandas as pd
import reslog
OUT = 'data/harness/offline_02d'
r = subprocess.run([sys.executable, 'harness/offline_check.py', '--out', OUT], capture_output=True, text=True,
                   timeout=900)
print(r.stdout[-8000:]); print(r.stderr[-3000:])
with open(f'{OUT}/offline_check.json') as f:
    RES = json.load(f)
RES += [dict(check=name, where='offline', passed=bool(ok), detail=str(value)) for name, ok, value in UNIT]
TAB = pd.DataFrame(RES)[['check', 'where', 'passed', 'detail']]
TAB.to_json('reports/harness/offline_check_02d.json', orient='records', indent=1)
DECISION = 'PASS' if (TAB.passed == True).all() else 'FAIL'
mark = lambda p: 'PASS' if p is True else ('SKIP' if p is None else 'FAIL')
TEXT = (f'DECISION: {DECISION} (offline checks O1-O14 and unit checks U1-U16, run in Colab; the warm homes, bytes per '
        f'host and the background download rule run on the VM in H1, V6, V7 and V8)\n\n'
        + '\n'.join(f'{mark(p)}  {c}: {d}' for c, p, d in zip(TAB.check, TAB.passed, TAB.detail)))
with open('reports/harness/decision_02d.txt', 'w') as f:
    f.write(TEXT + '\n')
reslog.log_result('02d warm homes: offline and unit checks', TEXT)
pd.set_option('display.max_colwidth', 400)
display(TAB)
assert DECISION == 'PASS', 'offline or unit checks failed: see the table'

## Stage C. Commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb02d: warm homes: benign runs start from a private copy of a per-version template home (returning user, not a first launch); homes.py makes, checks (Codex plugin catalogue sync finished, no API key in any file) and copies templates; the warm-up holds its first model call at the gateway (call_gateway.py --hold-until) until Codex has finished its plugin catalogue sync; vm_gate.py adds H1 warm homes, reports bytes up and down per host in V6, V7 and V8 and fails a run with a background download over 1,000,000 bytes; offline check O14 and unit checks U11-U16; README warm-home section; reports/harness/offline_check_02d.json, decision_02d.txt'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)